# VQCC Hedge Wizardry

**What this is.** An ongoing public collection of *small* verified tweaks to circuits for the [verifiable quantum circuit compilation competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation). A tweak lands here when it passes the local unmodified verifier and its expected gain is below +0.5% of the overall score. These gains are released straight away: the circuit in the [VQCC Hedge Wizardry dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry), the explanation here. The notebook then runs the verifier on Kaggle.

**How releases work.** Each new version of this notebook opens with what is new in it: the entry, the gain, and the technique it demonstrates, explained in full. Earlier entries are then summarised in a few lines each, with a link to the notebook version where they were the headline, for the full detail. A techniques index lists every technique shown so far. The [version history](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) of this notebook and the dataset is the dated record.

Each entry is:

- **explained**: what changed and why it pays, written for someone new to the task;
- **verified**: the unchanged official verifier passes the circuit, on Kaggle, in the cells at the end. For DQI the gain is also judged on its expected cost over 48 fixed random draws, never on one lucky draw;
- **credited**: most tweaks are made on someone else's public circuit, and that author is named first, with the licence kept.

**It is also a working submission notebook.** The hidden cells at the end take the cheapest public circuit for each task from our [VQCC public circuits dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits). Where an entry is cheaper for its task, they swap it in, check every file's SHA-256, run the unchanged official verifiers and write `submission.zip` with a receipt.

**How to read it.** The code is hidden; click a cell to see it. Numbers carry the evidence tier they were measured at, and a number is never shown at a higher tier than that:

- *static model*: arithmetic on the official cost formula;
- *local prescreen*: our re-implementation of the official evaluator's functions, used inside a search;
- *local engine*: our fast re-implementation of the simulator, cross-checked against the official verifier;
- *local official verifier*: the unmodified competition verifier on a local build of kickmix (*re-verified by us* when the circuit is someone else's published one);
- *official verifier (Kaggle)*: the same verifier run by this notebook on Kaggle.

Bigger changes, and the release-and-recompute method, are explained in [Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo).

In [ ]:
def rank_cost(r):
    # Cost used to rank a manifest row within its task (lower is better); None if unscored. A DQI cost at one salt is
    # one draw of its sampled failures (one or two phase errors move R from 1.19126 to 1.19673 / 1.20139), so DQI
    # ranks by dqi_expected_cost (mean over a 48-draw sweep) when known, else dqi_zero_failure_cost (R floor x Q^2 x
    # mean Toffolis), else local_task_score (older manifests). invsqrt and chemistry rank by local_task_score.
    keys = ("dqi_expected_cost", "dqi_zero_failure_cost", "local_task_score") if r.get("task") == "dqi" \
        else ("local_task_score",)
    for k in keys:
        try:
            v = float(r.get(k) or "nan")
        except ValueError:
            continue
        if v == v:
            return v
    return None

def rank_basis(r):
    # Which column rank_cost used, for the tables
    if r.get("task") != "dqi":
        return "default salt"
    return ("expected (48 draws)" if r.get("dqi_expected_cost") else
            "zero-failure" if r.get("dqi_zero_failure_cost") else "default salt")

# Tie rule: rows within REL_TIE (relative) of the task's cheapest ranking cost are a tie, and the tie goes to the
# earliest public circuit (the original). A byte-different copy of a circuit changes the verifier's measurement
# draw, so it can come out a hair cheaper by noise alone; it should not displace the original's designer.
import datetime as _dt
REL_TIE = 1e-5

def public_when(r):
    # First non-empty, readable public_since / first_known / release_date as UTC (date only = 00:00 UTC); else None
    for k in ("public_since", "first_known", "release_date"):
        s = (r.get(k) or "").strip()
        if not s:
            continue
        try:
            d = _dt.datetime.fromisoformat(s.replace("Z", "+00:00"))
        except ValueError:
            continue
        return d.replace(tzinfo=_dt.timezone.utc) if d.tzinfo is None else d.astimezone(_dt.timezone.utc)
    return None

def rank_order(rows):
    # Scored rows of one task: first the pick, by a replay in public order (earliest first; missing dates last,
    # then lowest cost, then sha256) in which a later row takes the pick only by beating it by more than REL_TIE
    # (so a tie never moves the pick, however other rows move the cheapest); then the rest of the tie group
    # (cost <= cheapest x (1 + REL_TIE)) in public order; then the others by cost, then sha256
    scored = [(rank_cost(r), r) for r in rows]
    scored = [(c, r) for c, r in scored if c is not None]
    if not scored:
        return []
    limit = min(c for c, _ in scored) * (1 + REL_TIE)
    far = _dt.datetime.max.replace(tzinfo=_dt.timezone.utc)
    def tie_key(cr):
        w = public_when(cr[1])
        return (w is None, w or far, cr[0], cr[1].get("sha256") or cr[1].get("sha12") or "")
    by_time = sorted(scored, key=tie_key)
    pick = by_time[0]
    for cr in by_time[1:]:
        if pick[0] > cr[0] * (1 + REL_TIE):
            pick = cr
    tied = [pick] + [cr for cr in by_time if cr is not pick and cr[0] <= limit]
    rest = sorted((cr for cr in scored if cr[0] > limit),
                  key=lambda cr: (cr[0], cr[1].get("sha256") or cr[1].get("sha12") or ""))
    return [r for _, r in tied + rest]

def pick_best(rows_for_task):
    # The row to use for a task: rank_order()[0], or None if no row is scored
    order = rank_order(rows_for_task)
    return order[0] if order else None


import base64, csv, glob, os, hashlib, json, shutil, sys, time, zipfile
from math import prod
from pathlib import Path

T0 = time.monotonic()
import numpy as np
import pandas as pd
from IPython.display import HTML, display

TASK_NAMES = {"invsqrt": "Inverse square root", "dqi": "DQI decoder", "chemistry": "Chemistry (dfthcsos)"}

def show(df, height=None):
    # A scrollable HTML table: readable on the notebook page, with live links
    html = df.to_html(index=False, render_links=True, escape=True, float_format=lambda x: f"{x:,.6g}")
    style = f"max-height:{height}px;overflow:auto;" if height else "overflow:auto;"
    display(HTML(f'<div style="{style}">{html}</div>'))

INPUT = Path(os.environ.get("VQCC_NB_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("VQCC_NB_WORK", "/kaggle/working"))
SUB = Path(os.environ.get("VQCC_NB_SUB", "/tmp/submission"))   # assembled here; only submission.zip and check.json go to Output
WHEEL_GLOB = "kickmix-*-manylinux*x86_64*.whl"
# Organisers' reference costs (the score is the geometric mean of reference / ours over the three tasks)
REFERENCE = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "chemistry": 3355599588.3155375}
VERIFIER_TASK = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}

def find(pattern, root=INPUT):
    hits = sorted(Path(p) for p in glob.glob(f"{root}/**/{pattern}", recursive=True))  # follows symlinked mounts
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {root}")
    return hits

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()

def family_sha256(directory):
    # The dataset's chemistry family hash: SHA-256 of sorted '<relative path>\t<file sha256>\n' lines
    files = sorted(p for p in Path(directory).rglob("*") if p.is_file())
    text = "".join(f"{p.relative_to(directory).as_posix()}\t{sha256(p)}\n" for p in files)
    return hashlib.sha256(text.encode()).hexdigest()

In [ ]:
# Figures live in the Hedge Wizardry dataset's figures/ folder.
def figure(name, caption=""):
    hits = sorted(Path(p) for p in glob.glob(f"{INPUT}/**/figures/{name}", recursive=True))
    if not hits:
        print(f"[figure {name} is not in the attached dataset version]")
        return
    mime = "image/gif" if name.endswith(".gif") else "image/png"
    data = base64.b64encode(hits[0].read_bytes()).decode()
    cap = f'<figcaption style="color:#52514e;font-size:13px;margin-top:4px">{caption}</figcaption>' if caption else ""
    display(HTML(f'<figure style="margin:10px 0"><img src="data:{mime};base64,{data}" '
                 f'style="max-width:100%;height:auto" alt="{name}"/>{cap}</figure>'))

## 1. New in this release (Version 26)

**HW-24 (2026-11-05, chemistry): Syed Asad Ali's eff6a4863f08 with eleven loaders re-planned.** Syed Asad Ali's public chemistry family `eff6a4863f08` with eleven of its angle loaders (slots 4–14) re-planned and emitted in the table's own pi gauge: 4,229 → 4,207 loader Toffolis per pass, 15,120 → 15,076 Toffolis, same 427 qubits and λ_eff. Local unmodified verifier PASS at 301,875,698.74 versus 302,756,736.86 (-0.2910%), about +0.10% overall by static arithmetic.

Techniques demonstrated:

- **Chained loader re-planning on a newly published family**: A new public table is taken through the loader tools the day it appears: read its storage convention (here a pi gauge with 10 flip patterns, and brot = 8 read from the circuits because the weights file has no brot field), compare each loader with the planner's floor, re-plan only the loaders above it, and chain the runs, each starting from the previous run's output, with fresh seeds for the slots still above the floor. A loader changes only if it is strictly cheaper and passes a native check on its real inputs.
- *Reused:* Loader re-planning as a repeatable step (dfthc_refit replan) (see the techniques index).
- *Reused:* Padding rows as don't-cares in the loader search (see the techniques index).
- *Reused:* Gauge-aware loader emission for a pi-gauge table (see the techniques index).

The full write-up follows: what improved and by how much, the technique explained, what it builds on, and how to reproduce or use the files. Earlier entries are summarised in section 2, with links to the notebook version where each was the headline.

### HW-24: eleven re-planned loaders on Syed Asad Ali's new chemistry family

On 2026-11-05 Syed Asad Ali published a new chemistry family, `eff6a4863f08` ([notebook](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) v4, Apache-2.0). Everything in it is their own: a new (8, 143, 44) word table, their own loader compiler and their own weights (they credit Heliosli for the refit up to the accuracy gate). At 302,756,736.86 it is cheaper than our HW-23 (303,694,816.97), so it **supersedes HW-23**. HW-24 keeps Syed Asad Ali's word table, weights and loaders 0–3 byte for byte, and replaces the other eleven angle loaders (slots 4–14) with cheaper ones.

**What the loaders do.** The angle words are read by 15 small lookup circuits (loaders), one per slot; the chemistry model charges each loader's Toffolis twice (two passes). Syed Asad Ali's compiler builds the coded loaders (slots 4–14) at 277 to 281 Toffolis each. By our reading of the files, on slots 4–9 it builds a smaller shared lookup cache (75 Toffolis instead of 80) and pays for it with more products (202 instead of 196). Our `dfthc_refit` tool's shared planner, which follows "i want to run gemma 9b"'s coded-loader construction, finds plans with a full cache and 196 products: **276 Toffolis** on ten of the eleven slots and 277 on slot 12. The 12 never-read padded rows are left free in the search (as in HW-18).

**The pi gauge again.** Syed Asad Ali's table is stored in the pi gauge introduced by Atakan Aldemir: on about 21% of the entries the top bit is flipped by a pattern set by the low address bits, which adds π to the stored angle and changes nothing in the chemistry. The gauge-aware emitter from HW-23 (built on Lera's (marculera) emitter) handles it. On slots 10, 12 and 13 the flip pattern changes between slots, so those words keep their top bit as an output; each of these loaders stays at its published width (127, 127 and 128 qubits, under the 128 cap). Every new loader passed a native check on its real inputs (the previous slot's words) and on random words with the incoming flip pattern.

**How the search ran.** Three chained runs, each starting from the previous run's output: slots 10–14 first, then 4–9, then slots 8 and 12 again with new seeds (slot 8 gained one more Toffoli; slot 12 found no 196-product plan). A loader is replaced only if it is strictly cheaper.

| | Syed Asad Ali `eff6a4863f08` | HW-24 `ec58b2f5fdcc` |
| --- | ---: | ---: |
| Loaders 4–9 (Toffolis per pass) | 277 each | **276 each** |
| Loaders 10 / 11 / 12 / 13 / 14 | 278 / 278 / 279 / 281 / 281 | **276 / 276 / 277 / 276 / 276** |
| Loaders 0–3, word table, weights | Syed Asad Ali | same, byte-identical |
| Loader qubits (cap 128) | 126–128 | unchanged per slot |
| All loaders (Toffolis per pass) | 4,229 | **4,207** |
| Whole chemistry model (Toffolis) / qubits | 15,120 / 427 | **15,076** / 427 |
| λ_eff | 46.893662 | 46.893662 (unchanged) |
| Full chemistry cost | 302,756,736.86 | **301,875,698.74** (-0.2910%) |

Every row is from the local official verifier (default salt); the cells below re-run it on Kaggle. The change is +0.0972% expected GM (arithmetic, with Syed Asad Ali's `19a0386ae02f` for invsqrt and HW-22 for DQI), not a leaderboard score. The weights are Syed Asad Ali's, so both accuracy errors are theirs too: each sits about 1e-5 mHa under its 20 mHa cap. That is why the hosted entry gate below is the check that matters.

The shared planner and the gauge-aware emitter are in our local `dfthc_refit` 1.6.0-dev; the public `dfthc_refit` utility script does not have them yet, so the eleven loaders cannot yet be regenerated on Kaggle. The circuits here can be used as they are.

**Credit.** Base: **Syed Asad Ali**'s public chemistry family `eff6a4863f08` ([notebook](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) v4, Apache-2.0): their own (8, 143, 44) word table, their own loader compiler and their own weights (they credit Heliosli for the refit up to the accuracy gate). The table, the weights (`submission.npz`) and loaders 0–3 are theirs, unchanged. The eleven new loaders (slots 4–14) come from our `dfthc_refit` shared planner, whose construction follows **i want to run gemma 9b**'s coded loaders, with Syed Asad Ali's block partition and pairing after **lightningv08**, and are emitted with a gauge-aware version of the emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `tensorqrom`, Apache-2.0; Lera's notice credits mrmeows and Heliosli), which handles the pi-gauge storage convention introduced by **Atakan Aldemir**. Plans and emission (our `dfthc_refit` 1.6.0-dev) by manwithacat (James Barlow), assisted by Claude (Anthropic).

**Files and reuse.** Copy `circuits/chemistry/ec58b2f5fdcc/` as `dfthcsos/`; family hash `ec58b2f5fdccef3d073eeb362322f95847868dbd65c684a44c36260fb1629528`. Evidence: `evidence/HW-24/verify_summary.json`, `metrics.json` (official metrics of Syed Asad Ali's family and HW-24) and `loaders.json` (per loader: source and run, products, Toffolis, qubits, plan and file hashes, the borrowed and kept top bits with their flip patterns, and the native check). Apache-2.0 notices are retained.

## 2. Earlier releases

Each earlier entry in brief: the gain at the time, the technique, the credit, and whether a newer public circuit has since overtaken it. Its circuit stays in the dataset, and the entry gate at the end still verifies it. For the full explanation, open the notebook version where it was the headline.

### HW-23, headline of Version 25 (2026-11-05, chemistry): HW-21's weights with four cheaper loaders, emitted in the table's pi gauge

HW-21's weights (on Heliosli's `aba448aeadb5`) with four of Atakan Aldemir's angle loaders (slots 8, 11, 12, 14) re-planned one Toffoli cheaper and emitted in the table's own pi gauge: 15,181 → 15,173 Toffolis, same 427 qubits and λ_eff. Local unmodified verifier PASS at 303,694,816.97 versus HW-21's 303,854,940.78 (-0.0527%; -0.0566% against `aba448aeadb5`), about +0.018% overall by static arithmetic.

- **Technique:** Gauge-aware loader emission for a pi-gauge table.
- **Credit:** Base: our HW-21 (dataset v23 / notebook v24), a weight refit of Heliosli's `aba448aeadb5` ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) v10, Apache-2.0). That family is **Heliosli**'s weight refinement of **Akmal Xodarev (busyaprime)**'s refit, on **Syed Asad Ali**'s (8, 143, 45) word table with **Atakan Aldemir**'s pi-gauge tree and angle loaders, integrated by **Kirill Tymchishin** (as credited in Heliosli's notebook). Words and 11 of the 15 loaders are theirs, unchanged; the weights are HW-21's. The four new loaders keep the planning model and emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `tensorqrom`, Apache-2.0); the shared planner's construction follows **i want to run gemma 9b**'s coded loaders and Syed Asad Ali's block partition, with pairing after **lightningv08**. Plans and gauge-aware emission (our `dfthc_refit` 1.5.0-dev) by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Released 2026-11-05 in dataset v24 / notebook v25; hosted entry gate PASS in notebook v25. Superseded on 2026-11-05 by Syed Asad Ali's public `eff6a4863f08` (302,756,736.86, cheaper; their own table, loaders and weights), on which HW-24 builds.
- **Full detail:** [Version 25](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/25) of this notebook.

### HW-21, headline of Version 24 (2026-11-05, chemistry): a weight refit of Heliosli's aba448aeadb5 at fixed words and loaders

A weight refit of Heliosli's public chemistry family `aba448aeadb5` (the current public best) with the words and all 15 loaders unchanged: λ_eff 46.876470 → 46.874651. Local unmodified verifier PASS at 303,854,940.78 versus 303,866,734.23 (-0.0039%, about +0.0013% overall by static arithmetic).

- **Technique:** Refit with the error margins held at the input's values.
- **Credit:** Base family `aba448aeadb5` by **Heliosli** ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) v10, Apache-2.0): Heliosli's weight refinement of **Akmal Xodarev (busyaprime)**'s refit, on **Syed Asad Ali**'s (8, 143, 45) word table with **Atakan Aldemir**'s pi-gauge tree and angle loaders, integrated by **Kirill Tymchishin** (as credited in Heliosli's notebook). Words and all 15 loaders are theirs, unchanged. Weight refit (three chained runs of our `dfthc_refit` tool, error margins held) by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Released 2026-11-05 in dataset v23 / notebook v24; hosted entry gate PASS in notebook v24. Superseded on 2026-11-05 by HW-23 (the same weights with four cheaper loaders).
- **Full detail:** [Version 24](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/24) of this notebook.

### HW-22, headline of Version 24 (2026-11-05, dqi): two public 891-qubit DQI decoders joined where they are byte-identical

The unsynthesis section of busyaprime's public DQI decoder `fd0ef09efd45` joined to the rest of the current public pick `1c43b7ff2fcf` (Alex Chilton's version, released by Leo Tang), at a stretch where the two are byte-identical: 14 fewer static Toffolis, same 891 qubits, all 48 draws PASS; 48-draw expected cost -0.0035% (about +0.0012% overall by static arithmetic). Local unmodified verifier PASS at the default salt.

- **Technique:** Joining two public circuits where they are byte-identical.
- **Credit:** Unsynthesis section from `fd0ef09efd45` by **Akmal Xodarev (busyaprime)** ([33 QCC V8 members](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) v10, Apache-2.0), their exact rewrite of **Syed Asad Ali**'s 891-qubit decoder `fc3ba81614e1`. Everything else, including the unextract section, from `1c43b7ff2fcf`: **Alex Chilton**'s decoder version, released by **Leo Tang** in [VQCC T7421 Submission](https://www.kaggle.com/code/leotang09/vqcc-t7421-submission) v17 (Apache-2.0), on the public 891-qubit decoder line (Syed Asad Ali, Agniv Banerjee and others, as credited there). The join of the two public circuits and its verification are by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Released 2026-11-05 in dataset v23 / notebook v24; hosted entry gate PASS in notebook v24.
- **Full detail:** [Version 24](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/24) of this notebook.

### HW-19, headline of Version 23 (2026-11-04, chemistry): the HW-18 padded-row step on Syed Asad Ali's new word table

The HW-18 padded-row step applied to Syed Asad Ali's new public family `f4291619f2fd` (their own (8, 143, 45) word table): the 11 coded loaders each save two products (282 → 280 Toffolis per pass), loaders 0–3 kept as shipped, words and weights unchanged. Local unmodified verifier PASS at 306,723,130.32 versus 307,606,537.91 (-0.287%, about +0.096% overall by static arithmetic).

- **Technique:** .
- **Credit:** Word table, angle loaders 0-3 and weights of family `f4291619f2fd` by **Syed Asad Ali** ([dataset](https://www.kaggle.com/datasets/syed3000/vqcc-own-circuit-portfolio) v3 and [notebook](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) v3, Apache-2.0; their own fit and rounding of an (8, 143, 45) table in Lera's layout); angle-loader search code (`kernel_search.py`, run unmodified) by **qwertykeypadapple** ([notebook](https://www.kaggle.com/code/qwertykeypadapple/qcc-kernel-search-24-65) v1 and its dataset, Apache-2.0); the emitter by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, Apache-2.0). Padded-row re-planning of loaders 4-14 by manwithacat (James Barlow), assisted by Claude (Anthropic), as in HW-18.
- **Status (2026-11-05):** Released 2026-11-04 in dataset v22 / notebook v23; hosted entry gate PASS in notebook v23. Superseded on 2026-11-05 by newer public chemistry families: i want to run gemma 9b's `6566cd395128` (305,652,489.65), then Heliosli's `aba448aeadb5` (303,866,734.23), the base of HW-21.
- **Full detail:** [Version 23](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/23) of this notebook.

### HW-18, headline of Version 22 (2026-11-04, chemistry): one product fewer per loader, from the table's never-read padding rows

qwertykeypadapple's kernel search re-run on their own public family `a344f422f2e0`, but with the 12 never-read padded table rows left free: each of the 15 loaders saves one product (4,303 → 4,288 Toffolis per pass), words and weights unchanged. Local unmodified verifier PASS at 308,604,860.13 versus 309,209,296.10 (-0.195%, about +0.065% overall by static arithmetic).

- **Technique:** Padding rows as don't-cares in the loader search.
- **Credit:** Angle-loader search code (`kernel_search.py`, run unmodified) and the shipped loaders of family `a344f422f2e0` by **qwertykeypadapple** ([notebook](https://www.kaggle.com/code/qwertykeypadapple/qcc-kernel-search-24-65) v1 and its dataset, Apache-2.0); weights by **Agniv Banerjee** (family `a56ec354e8b8`, Apache-2.0); the (8, 143, 45) table and the emitter by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, Apache-2.0). Padded-row choice and re-emission of all 15 loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Released 2026-11-04 in dataset v21 / notebook v22 (dataset v20 / notebook v21 failed: subdirectories missing from the upload); hosted entry gate PASS in notebook v22. Superseded the same evening by Syed Asad Ali's public f4291619f2fd.
- **Full detail:** [Version 22](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/22) of this notebook.

### HW-17, headline of Version 20 (2026-11-04, chemistry): the HW-16 loaders on Isabe11aM00re's newer family, via dfthc_refit replan

`dfthc_refit replan` run on Isabe11aM00re's new public family `35694008ca5e` (her new loader 10 and weights on Ayodeji's loaders, Lera's (8, 143, 45) table): it keeps her loaders 4–14 and swaps in our re-planned loaders 0–3 (303 Toffolis instead of 307), words and weights unchanged. Local unmodified verifier PASS at 310,780,568.95 versus 311,425,300.12 (−0.207%) and versus HW-16's 310,821,125.36 (−0.013%, about +0.004% overall by static arithmetic).

- **Technique:** Loader re-planning as a repeatable step (dfthc_refit replan).
- **Credit:** Angle loaders 4–14 by **Ayodeji** ([notebook](https://www.kaggle.com/code/ayodejiibrahimlateef/vqcc-24-56308-local) v4, family `0ed964465654`) with loader 10 and the weights by **Isabe11aM00re** ([notebook](https://www.kaggle.com/code/isabe11am00re/qcc-v66s33w-submission) v2, family `35694008ca5e`), on the (8, 143, 45) table and emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `4ae679db726d`); all Apache-2.0. Loaders 0–3 re-planned by manwithacat (James Barlow) with our qrom-plan relation planner (as HW-16), assisted by Claude (Anthropic), and emitted with Lera's unmodified emitter.
- **Status (2026-11-05):** Released 2026-11-04 in dataset v18 / notebook v20; hosted entry gate PASS. Superseded the same evening by qwertykeypadapple's public a344f422f2e0.
- **Full detail:** [Version 20](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/20) of this notebook.

### HW-16, headline of Version 19 (2026-11-04, chemistry): four cheaper plain loaders (307 to 303 Toffolis) on the C = 45 frontier family

Our relation planner rebuilds the four plain angle loaders of the best public chemistry family (`a56ec354e8b8`: Agniv Banerjee's weights on Ayodeji's loaders, Lera's (8, 143, 45) table) at 303 Toffolis instead of 307, words and weights unchanged: local unmodified verifier PASS at 310,821,125.36 versus 311,465,857.07 (−0.207% chemistry, about +0.069% overall by static arithmetic).

- **Technique:** Re-planning a public loader's GF(2) relations from its own words.
- **Credit:** Angle loaders 4–14 by **Ayodeji** ([notebook](https://www.kaggle.com/code/ayodejiibrahimlateef/vqcc-24-56308-local) v4, family `0ed964465654`) and weights by **Agniv Banerjee (SomeJoint)** ([qcc-ladder-parts dataset](https://www.kaggle.com/datasets/somejoint/qcc-ladder-parts) v8, family `a56ec354e8b8`), on the (8, 143, 45) table and emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `4ae679db726d`); all Apache-2.0. Loaders 0–3 re-planned by manwithacat (James Barlow) with our qrom-plan relation planner, assisted by Claude (Anthropic), and emitted with Lera's unmodified emitter.
- **Status (2026-11-05):** Released 2026-11-04 in dataset v17 / notebook v19; hosted entry gate PASS (kickmix-engine).
- **Full detail:** [Version 19](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/19) of this notebook.

### HW-15, headline of Version 17 (2026-11-03, chemistry): constrained weight refit with dfthc_refit on SomeJoint's 15,585-Toffoli family

Our public dfthc_refit 1.1.0 tool, run on Agniv Banerjee’s new best public family `4a9adf368a10` (words and loaders unchanged): λ_eff 47.025650 → 47.023457, local unmodified verifier PASS at 313,664,330.04 versus 313,678,954.10 (−0.0047% chemistry, about +0.0016% overall by static arithmetic).

- **Technique:** A reusable constrained refit tool (dfthc_refit).
- **Credit:** Angle loaders and starting weights by **Agniv Banerjee (SomeJoint)** ([qcc-ladder-parts dataset](https://www.kaggle.com/datasets/somejoint/qcc-ladder-parts) v4, family `4a9adf368a10`); the (9, 143, 39) C = 39 table by **Syed Asad Ali** ([notebook](https://www.kaggle.com/code/syed3000/vqcc-24-43-portfolio) v1, `b9b05e14dbdb`), built on **Lera (marculera)**'s DF-THC compact-groups pipeline ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian)); all Apache-2.0. The weight refit by manwithacat (James Barlow) with our public [dfthc_refit](https://www.kaggle.com/code/manwithacat/dfthc-refit) 1.1.0 tool, assisted by Claude (Anthropic); its projected L-BFGS step follows an idea from anazemcev's public Apache-2.0 refit.
- **Status (2026-11-05):** Released 2026-11-03 in dataset v15 / notebook v17; hosted entry gate PASS (kickmix-engine).
- **Full detail:** [Version 17](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/17) of this notebook.

### HW-14, headline of Version 15 (2026-11-02, chemistry): dead-word flips that unlock 296-Toffoli angle loaders (Heliosli's weights)

Thirteen dead angle words (rotations on zero-amplitude branches, so every accuracy metric is unchanged) set to values that unlock one more paired relation on five loaders: 297 → 296 Toffolis each, 15,787 → 15,777 in total. With Heliosli’s newer public weights, the local unmodified verifier passed at 317,574,154.53 versus 317,775,443.85 for the best public family (−0.063% chemistry, about +0.021% overall by static arithmetic).

- **Technique:** Dead-word flips for QROM loaders.
- **Credit:** C40 table, words and the 128-qubit loader emitter (`tensorqrom.build_circuit`) by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); the kernel-relation compiler ideas and quotient pairing (`kernel_planes.py`, `permuted_kernel.py`, `rank_parity.py`) from **lightningv08** (Vladislav Lavrentev), whose NOTICE credits **Ahmed Berat Özer** (generic kernel-relation emitter) and **mrmeows** (paired-product kernel, prefix cache, HMR emitter); the extended-relation idea and formula T_k = 89 + (525 − rel)/2 from **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class)); weights by **Heliosli** ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) v42); all Apache-2.0. The dead-word flip search (Rust qrom-plan screen/stack/batch), relation plans and the rebuilt loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Published (dataset v14, notebook v15) and hosted official verifier PASS on 2026-11-03.
- **Full detail:** [Version 15](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/15) of this notebook.

### HW-13, headline of Version 14 (2026-11-02, chemistry): QROM loop: block-design relation plans and a layout search for all 15 angle loaders

The QROM loop on Lera’s C40 loaders: block-design relation plans with exchange repair take loaders 0–3 from 321 to 314 Toffolis, and a layout search takes loaders 4–14 from 298 to 297 (15,861 → 15,787 in total). With SomeJoint’s newer public weights, the local unmodified verifier passed at 317,777,218.02 versus HW-12’s 319,274,986.45 (−0.47% chemistry, about +0.16% overall by static arithmetic).

- **Technique:** Block-design relation plans with exchange repair.
- **Credit:** C40 table, words and the 128-qubit loader emitter (`tensorqrom.build_circuit`) by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); the kernel-relation compiler ideas and quotient pairing (`kernel_planes.py`, `permuted_kernel.py`, `rank_parity.py`) from **lightningv08** (Vladislav Lavrentev), whose NOTICE credits **Ahmed Berat Özer** (generic kernel-relation emitter) and **mrmeows** (paired-product kernel, prefix cache, HMR emitter); the extended-relation idea and formula T_k = 89 + (525 − rel)/2 from **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class)); weights by **SomeJoint** (Agniv Banerjee’s team; [dataset](https://www.kaggle.com/datasets/somejoint/qcc-ladder-parts) v1); all Apache-2.0. The relation plans (HW-12 search, block designs, exchange repair, layout search) and the rebuilt loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Published (dataset v13, notebook v14) and hosted official verifier PASS on 2026-11-02.
- **Full detail:** [Version 14](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/14) of this notebook.

### HW-12, headline of Version 13 (2026-11-02, chemistry): angle loaders recompiled with a systematic GF(2) kernel-relation search (all 15 loaders)

All 15 angle loaders of Lera’s C40 table rebuilt with her unmodified 128-qubit emitter from plans with more GF(2) kernel relations: loaders 4–14 drop to 298–299 Toffolis (from 301–310), total 16,071 → 15,861. Local unmodified verifier PASS at 319,274,986.45 versus the public assembly’s 323,502,194.52 (−1.31% chemistry, about +0.44% overall score by static arithmetic).

- **Technique:** Systematic GF(2) kernel-relation search for QROM loaders.
- **Credit:** C40 table, words and the 128-qubit loader emitter (`tensorqrom.build_circuit`) by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); the kernel-relation compiler ideas and quotient pairing (`kernel_planes.py`, `permuted_kernel.py`, `rank_parity.py`) from **lightningv08** (Vladislav Lavrentev), whose NOTICE credits **Ahmed Berat Özer** (generic kernel-relation emitter) and **mrmeows** (paired-product kernel, prefix cache, HMR emitter); the extended-relation idea and formula T_k = 89 + (525 − rel)/2 from **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class)); weights by **sigmaborov** ([notebook](https://www.kaggle.com/code/sigmaborov/22-85569-lb-exact-gauge-search) v15); all Apache-2.0. The relation search and the rebuilt loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Published (dataset v12, notebook v13) and hosted official verifier PASS on 2026-11-02.
- **Full detail:** [Version 13](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/13) of this notebook.

### HW-11, headline of Version 12 (2026-11-02, chemistry): HW-10’s weights on the cheapest public C40 angle loaders (matousakura’s loaders 0–3)

HW-10’s weights on the cheapest public loaders, including matousakura’s recompiled loaders 0–3: local unmodified verifier PASS at 324,791,746.23 versus the public assembly’s 324,815,061.65 (−0.0072% chemistry, about +0.0024% overall by static arithmetic).

- **Technique:** Weights and loaders combined across entries.
- **Credit:** C40 table and words by **Lera (marculera)**; angle loaders 0–3 recompiled by **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class) v8) with Lera’s public 5-word-QROM compiler, loaders 4–14 by **Dmitrii Gluzdov** and Lera; weights are our HW-10 refit of **yoyobar**’s weights (after refits by Isabe11aM00re, Heliosli, Ahmed Berat Özer and Alex Chilton); all Apache-2.0. Combination by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Published (dataset v11, notebook v12) and hosted official verifier PASS on 2026-11-02.
- **Full detail:** [Version 12](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/12) of this notebook.

### HW-10, headline of Version 11 (2026-11-02, chemistry): weights-only constrained refit of yoyobar’s public C40 chemistry

On yoyobar’s public C40 chemistry `a317079936e9` (weights on Gluzdov’s and Lera’s loaders and Lera’s table), our weights-only constrained refit lowered λ_eff 47.035278 → 47.031870 by letting `error_1` use its headroom (13.27 → 14.75 mHa). The local unmodified verifier passed at 326,764,450.98 versus 326,788,128.83 (−0.0072% chemistry cost, about +0.0024% overall score by static arithmetic). Words and all 15 loaders are byte-identical.

- **Technique:** Constrained λ_eff refit at fixed words, e1 released.
- **Credit:** C40 factorization and 8-bit words by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); angle loaders 0-3 and 14 by **Dmitrii Gluzdov** ([notebook](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source)), the rest Lera's; base weights by **yoyobar** ([notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v16), after refits by Isabe11aM00re, Heliosli, Ahmed Berat Özer and Alex Chilton; all Apache-2.0. Change: the constrained W/H2_I refit by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **Status (2026-11-05):** Published (dataset v10, notebook v11) and hosted official verifier PASS on 2026-11-02; its portfolio was submitted to the competition at 19:10 UTC on the user's request.
- **Full detail:** [Version 11](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/11) of this notebook.

### HW-09, headline of Version 10 (2026-11-02, dqi): conditionally clean S12/S6 flag computations on busyaprime’s public DQI

Twelve 12-input AND computations and twelve 6-input AND computations in busyaprime's public decoder are replaced with our exhaustively checked conditionally clean templates. All 24 sites have proved clean-wire obligations. Same 892 qubits; -293.998 mean Toffolis per shot; -0.07351% DQI expected cost on 48 shared draws, cheaper on every draw with identical errors. Expected portfolio GM 23.90474283 to 23.91060278 (+0.02451%, arithmetic). Unmodified local official verifier PASS; hosted official verifier PASS in notebook v10, exact cost match.

- **Technique:** Conditionally clean flag computation with measurement-based cleanup.
- **Credit:** Base by **Akmal Xodarev (Busya PRIME)**: [33 QCC V8 members](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) v9, from [Exact DQI Rewrites](https://www.kaggle.com/code/busyaprime/23-910-local-gm-exact-dqi-rewrites-892-qubits) v15, Apache-2.0. The core decoder is **SomeJoint (Agniv Banerjee)**'s q26 892-qubit design; busyaprime adds exact product caching, credits **Ahmed Berat Özer** for that idea, and incorporates our HW-07 ladder. Our S12/S6 templates were developed in the earlier local relay, following **Khattar and Gidney**'s conditionally clean ancilla method; this port and verification are by **manwithacat (James Barlow), assisted by Codex (OpenAI)**.
- **Status (2026-11-05):** Published in Hedge dataset v9 and notebook v10; hosted official verifier PASS, exact cost match. Version 9 artifacts omitted HW-09 through a stale dataset attachment; v10 explicitly pinned dataset v9 and required the headline SHA before verification. Hosted ZIP SHA-256 cff190d21377b7ec425a1baac2ac68f971e180eb66f735dd95fb5e672d041ad7. No competition submission.
- **Full detail:** [Version 10](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/10) of this notebook.

### HW-08, headline of Version 8 (2026-11-01, dqi): measurement-based uncomputation for the 12 three-input flag ANDs and the 12 six-input normalize flags, ported to SomeJoint's 892-qubit DQI

SomeJoint's (Agniv Banerjee) newer public 892-qubit DQI decoder `b7ca9bce749d` overtook HW-07 on the public frontier. It still clears its 12 three-input flag ANDs and 12 six-input normalize flags by re-running borrowed-wire Toffoli chains, so we carried both HW-06's and HW-07's measurement-based uncomputation onto it. Our census tool found all 24 sites in about a minute. Result `5e86f69a9611`: same 892 qubits, −207.01 mean Toffolis per shot, −0.0514% DQI cost over 48 fixed draws, cheaper on every draw, with the same 7 failing shots (local engine); about +0.017% expected overall score. Local official verifier PASS at the default salt. **Published as a reference record:** before release, busyaprime's public exact product caching on the same SomeJoint base (`3779a3431e9c`) overtook it by 0.60% on the same 48 draws; that circuit already clears the HW-06-class sites by measurement-based uncomputation, and our census no longer finds the HW-07-class sites there. The value of this entry is the worked port: what the circuit does, which sites changed, and the steps taken.

- **Technique:** Porting a tweak to a new base by site census.
- **Credit:** base circuit `b7ca9bce749d` by **SomeJoint (Agniv Banerjee)** ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi) v26, Apache-2.0): GPT-6 Astra Pro's q26 decoder round, with Claude putting SomeJoint's flag carry, which follows **Ali Haydar Özdağ**'s, on it; the flag carry is as in `bdee18489fac` (**yoyobar**'s one-carry history). Change: HW-06's and HW-07's measurement-based uncomputation at 24 sites, found by our census tool, by manwithacat.
- **Status (2026-11-05):** Headline of Version 8, published as a reference record. Overtaken before release by busyaprime's public exact product caching on the same SomeJoint base, `3779a3431e9c` (48-draw mean 379,530,489,015; -0.60% against HW-08 on the same draws), which also clears the HW-06-class sites (12 self-uncompute Toffolis) by measurement-based uncomputation; our census finds the HW-07-class sites no longer present there. Local official verifier PASS at the default salt; the entry gate in this version is its first hosted run.
- **Full detail:** [Version 8](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/8) of this notebook.

### HW-07, headline of Version 7 (2026-11-01, dqi): measurement-based uncomputation with a two-wire ladder for the 12 six-input normalize flags in yoyobar's 892-qubit DQI

On our HW-06 circuit (yoyobar's public 892-qubit DQI `6d58bd6d0232` with 12 flag ANDs made measurement-based), we clear the six-input pivot flag of each epoch's normalize step by measurement-based uncomputation instead of a 16-Toffoli chain on borrowed wires. When the measurement gives 1, the phase fix rebuilds the six-input AND on a ladder of two clean wires, about 2.25 Toffolis expected per site. Result `1779b4e5967f`: same 892 qubits, −164.95 mean Toffolis per shot, −0.0409% DQI cost over 48 fixed draws against HW-06 and −0.0513% against `6d58bd6d0232`, with the same 7 failing shots (local engine); about +0.014% expected overall score against HW-06 (+0.017% against `6d58bd6d0232`). Local official verifier PASS at the default salt.

- **Technique:** Measurement-based uncomputation of a multi-input AND with a clean-wire ladder.
- **Credit:** base circuit `c253111974fc`, our HW-06, on **yoyobar**'s `6d58bd6d0232` (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Reading Ali Haydar Özdağ's (ahoirg) public reproducible rebuild of SomeJoint's decoder sources ([notebook](https://www.kaggle.com/code/ahoirg/score-23-64254-reproducible-circuit-portfolio) v1) explained the normalize step's wires. Change: the two-wire measurement-based uncomputation of the 12 six-input flags, by manwithacat.
- **Status (2026-11-05):** Published on 2026-11-01: the notebook was pushed twice with identical content, as Versions 6 and 7 (a duplicate push), so Version 7 is its headline. Hosted official verifier PASS at the default salt in Version 6 (384,137,374,413.52, equal to `entries.csv`). Overtaken on the public frontier the same day by SomeJoint's (Agniv Banerjee) `b7ca9bce749d`: 48-draw mean cost 382,020,234,789.40 against HW-07's 382,637,339,169.98 (local engine, same draws). HW-08 carries HW-06's and HW-07's techniques onto `b7ca9bce749d`.
- **Full detail:** [Version 7](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/7) of this notebook.

### HW-06, headline of Version 5 (2026-11-01, dqi): measurement-based uncomputation for the 12 borrowed-wire flag ANDs in yoyobar's 892-qubit DQI

On yoyobar's public 892-qubit DQI circuit `6d58bd6d0232` (on SomeJoint's flag carry), our measurement-based uncomputation replaces 12 four-Toffoli uncomputes that borrowed an in-use wire. Result `c253111974fc`: same 892 qubits, −42.0 mean Toffolis per shot, −0.0104% DQI cost over 48 fixed draws with the same 7 failing shots (local engine), about +0.0035% expected overall score. Local official verifier PASS at the default salt.

- **Technique:** Measurement-based uncomputation of a borrowed-wire AND; Sound value tracking for provably-zero targets.
- **Credit:** base circuit `6d58bd6d0232` by **yoyobar** (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Change: measurement-based uncomputation at the 12 borrowed-wire flag ANDs, and the value tracker that found them, by manwithacat.
- **Status (2026-11-05):** Published on 2026-11-01 as the headline of Version 5; its hosted run of the unchanged official verifier on Kaggle passed (cost 382,537,501,561.76, 0 errors, equal to `entries.csv`), and it was that version's DQI pick. Superseded for DQI by HW-07 (`1779b4e5967f`), which builds on it: −0.0409% DQI cost over the same 48 draws (local engine).
- **Full detail:** [Version 5](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) (open the version list and choose Version 5) of this notebook.

### HW-05, headline of Version 4 (2026-11-01, chemistry): C40 open-pattern chemistry: fixed-word constrained refit

On Lera’s (marculera) public open-pattern C40 family `b163954eb5ac`, our weights-only constrained refit lowered λ_eff 47.105858 → 47.048390. The local unmodified verifier passed at 326,959,774.66 versus 327,359,144.12 (−0.1220% chemistry cost), about +0.0407% overall score by static arithmetic. The 8-bit words and all 15 angle loaders are byte-identical to Lera’s.

- **Technique:** Fixed-e1 constrained refit on an open-pattern C40 table.
- **Credit:** C40 factorization, 8-bit open-pattern words and all 15 loaders by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9, Apache-2.0). Change: the fixed-word constrained W/H2_I refit by manwithacat (James Barlow), assisted by Codex (OpenAI). The refit method was developed in HW-03 and applied to C42 in HW-04.
- **Status (2026-11-05):** Published and hosted-verifier PASS on 2026-11-01. Overtaken by Heliosli’s public chemistry: `0d75ed93ce40` (326,934,192.97) in the same hosted run, then `ffcc1ee812ac` (326,902,029.99, Heliosli’s notebook v36), and a public parts assembly of Heliosli’s `ffcc1ee812ac` weights with angle loaders that include Dmitrii Gluzdov’s (326,821,497.37). All three costs are our local official verifier runs; HW-05 is 326,959,774.66.
- **Full detail:** [Version 4](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364542669) of this notebook.

### HW-04, headline of Version 3 (2026-10-31, chemistry): C42 chemistry weights: refit spending e1 slack, alternated with the column gauge

Weights only, on Lera's (marculera) C42 table `e9a9289ab06e`: our refit spent the unused e2 and E_gap margin and raised e1 in steps to 19.9999 mHa, alternated with the column gauge. λ_eff 47.080918 → 46.976679, −0.2214% chemistry cost (local official verifier PASS), about +0.074% overall score. Angle loaders and word table byte-identical to Lera's.

- **Technique:** Slack-spending constrained refit (stepped e1 continuation); KKT shadow prices of the error margins.
- **Credit:** C42 factor/word table and loaders by **Lera (marculera)** (e9a9289ab06e, [notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v6, Apache-2.0). Change: the e1-spending refit + gauge alternation, by manwithacat. (The method was first run on Ahmed Berat Özer's C43 polish e9eb9336a5ec.)
- **Status (2026-11-05):** Overtaken by Lera’s public C40 open-pattern family `b163954eb5ac` and this HW-05 refit (local verifier cost 326,959,774.66). HW-04 remains a separate C42 demonstration.
- **Full detail:** [Version 3](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) (open the version list and choose Version 3) of this notebook.

### HW-03, headline of Version 2 (2026-10-31, chemistry): chemistry weights: constrained refit alternated with the column gauge

Weights only, on the public chemistry family `ec11a966fb64` (Ahmed Berat Özer's weights in Isabe11aM00re's per-rank column gauge): our constrained refit alternated with the column gauge took λ_eff from 48.293182 to 48.274616, −0.0384% chemistry cost (local official verifier PASS), about +0.013% overall score. Angle loaders and word table byte-identical.

- **Technique:** Constrained λ_eff refit at fixed words; Refit alternated with a per-rank orthogonal column gauge.
- **Credit:** weights by **Ahmed Berat Özer** (4eabe756bc36, [notebook](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), Apache-2.0); per-rank column-gauge idea and the base point ec11a966fb64 by **Isabe11aM00re** ([notebook](https://www.kaggle.com/code/isabe11am00re/qcc-v47-submission) v3, Apache-2.0); word table by **Atakan Aldemir**; the (8, 143, 60) factorization by **SomeJoint (Agniv Banerjee)**; loaders by **Heliosli, Dmitrii Gluzdov and Atakan Aldemir**. Change: the refit + gauge alternation and the gauge search, by manwithacat.
- **Status (2026-11-05):** Overtaken: Lera's (marculera) C42 table `e9a9289ab06e` (331,760,020, re-verified by us) is 1.7% cheaper than HW-03, and HW-04 (section 1) applies the same method to it.
- **Full detail:** [Version 2](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) (open the version list and choose Version 2) of this notebook.

### HW-01, headline of Version 1 (2026-10-30, dqi): 889 -> 888 qubits on yoyobar's DQI by release-and-recompute

On yoyobar's public 889-qubit DQI decoder `73e1b187b288`, our release-and-recompute freed one qubit for +460.4 mean Toffolis per shot (a qubit is worth about 960 here). Result `5faba4591853`: 888 qubits, −0.117% DQI cost at the default salt and −0.108% over 48 fixed draws (local official verifier; local engine), about +0.036% expected overall score. One preregistered threshold was missed (8 failing draws of 48 against the base's 7) and the release said so.

- **Technique:** Release-and-recompute (spooky pebbling); Liveness ledger to find the idle hold.
- **Credit:** base circuit `73e1b187b288` by **yoyobar** ([notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v8, Apache-2.0), on the decoder line of **SomeJoint (Agniv Banerjee)**; releasing a pole flag and recomputing it was first used publicly by **heliosli** on SomeJoint's decoder. Change: release-and-recompute (Quantum Voodoo line), the q852 release, the q878 reset and the handover erase, by manwithacat.
- **Status (2026-11-05):** Overtaken: newer public 888-qubit DQI circuits are cheaper, for example yoyobar's `25db915d86ab` and `e20e6abea7e2` and Ahmed Berat Özer's `fe98677c61ef` (398,268,953,061 at the default salt, −0.93% against HW-01; re-verified by us with the local official verifier, 2026-10-31).
- **Full detail:** [Version 1](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364036151) of this notebook.

## 3. Techniques index

Every technique demonstrated so far, where it first appeared, and which later entries reuse it.

| Technique | First shown | Reused in | What it does |
| --- | --- | --- | --- |
| Release-and-recompute (spooky pebbling) | HW-01, [Version 1](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364036151) | — | Measurement-based uncomputation (HMR) releases a value that sits idle across every peak run; it is recomputed (rematerialised) where it is next needed and the phase correction applied, so the peak qubit count falls for a few hundred extra Toffolis. Credit: ours (Quantum Voodoo line); releasing a pole flag was first used publicly by heliosli. |
| Liveness ledger to find the idle hold | HW-01, [Version 1](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364036151) | — | A liveness analysis (render.py ledger) lists each value's idle time at peak width, priced at the qubit's shadow price; it named the target (q852) before anything was built. Credit: ours (kickmix-macos fork). |
| Constrained λ_eff refit at fixed words | HW-03, [Version 2](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | HW-04, HW-05, HW-10, HW-11, HW-15 | Projected L-BFGS on log λ_eff over the continuous weights and offsets, with e1, e2 and E_gap held as equality constraints just inside their limits and a Gauss–Newton step back onto them after every move; the angle words and angle loaders, so qubits and Toffolis, never change. Credit: ours (constraint-manifold retraction idea from anazemcev's projected L-BFGS). |
| Refit alternated with a per-rank orthogonal column gauge | HW-03, [Version 2](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | HW-04, HW-15, HW-21 | Rotating one rank's columns by an orthogonal matrix leaves e1, e2 and E_gap unchanged but changes the column L1 norms in Λ; exact Jacobi sweeps plus a smoothed-L1 L-BFGS find a sparser frame between refits. Credit: gauge idea by Isabe11aM00re; the alternation and gauge search ours. |
| Slack-spending constrained refit (stepped e1 continuation) | HW-04, [Version 3](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | — | From a point that leaves error margin unused, the refit spends the e2 and E_gap margin and raises the e1 target in steps (16.5 → 18 → 19 → 19.9999 mHa), each from the previous point, with column gauges between; the extra e1 room lets the offsets lower the one-body norm Σ|eig h1′|. Credit: ours. |
| KKT shadow prices of the error margins | HW-04, [Version 3](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | — | The Lagrange multipliers of the equality-held constraints price each margin in % of λ_eff per mHa, showing which margin pays (here e2, about 200 times e1 at the final point) before compute is spent. Credit: ours (standard sensitivity analysis). |
| Fixed-e1 constrained refit on an open-pattern C40 table | HW-05, [Version 4](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364542669) | — | Move continuous W and H2_I on the e1/e2/E_gap constraint surface while freezing Lera’s integer words and all compiled loaders; a nonzero projected KKT residual predicts local room to improve λ_eff. Credit: our application of the HW-03 constrained refit to Lera’s C40 table. |
| Measurement-based uncomputation of a borrowed-wire AND | HW-06, [Version 5](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | HW-07, HW-08, HW-09 | A three-input AND computed with the help of a borrowed (dirty) wire costs 4 Toffolis to uncompute the same way. Measuring the result in the X basis instead clears it for free; the phase it leaves, (−1)^(m·abc), is removed by one CCZ on the three inputs, run only when the measured bit m is 1 (about half the shots). Credit: the construction is Gidney's (2018) measurement-based uncomputation of a temporary logical-AND; its application to these 12 sites is ours. |
| Sound value tracking for provably-zero targets | HW-06, [Version 5](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | — | A conservative tracker follows which wires are certainly 0 after each operation. A Toffoli whose target is provably 0 just afterwards is an uncompute, and so a candidate for measurement-based uncomputation. In this circuit it found exactly 12 such Toffoli groups; every other clean uncompute was already measurement-based. Credit: ours (harness probe, 2026-11-01). |
| Measurement-based uncomputation of a multi-input AND with a clean-wire ladder | HW-07, [Version 7](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/7) | HW-08 | A flag holding the AND of six bits, cleared by re-running a 16-Toffoli borrowed-wire chain, is instead measured in the X basis. Only when the outcome is 1 is a phase fix needed, (−1)^(y1⋯y6): it is applied by rebuilding the AND on two clean wires (3 CCX and one CCZ), each partial product then itself cleared by measurement-based uncomputation with a CZ or CCZ fix-up conditioned on its outcome. About 2.25 Toffolis expected per site instead of 16. Credit: measurement-based uncomputation of a temporary logical-AND is Gidney's (2018); the ladder and its application to these 12 sites are ours. |
| Porting a tweak to a new base by site census | HW-08, [Version 8](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/8) | — | A tweak found on one circuit often applies to a newer circuit from the same line. A census of the circuit file lists every site with the tweak's pattern (here, the three-input flag ANDs and six-input flags cleared by re-running a borrowed-wire chain) in about a minute, and the edits are generated from that list, so the port is mechanical and every site is accounted for. Re-running the original build code on the ported file and getting the same bytes checks the port. Credit: ours. |
| Conditionally clean flag computation with measurement-based cleanup | HW-09, [Version 10](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/10) | — | A clean target and one or two clean scratch wires replace a 6- or 12-input dirty-borrow compute chain; input wires become scratch only in the branch where their values are known, are restored, and temporary products are measured with phase corrections. Credit: Khattar and Gidney for conditionally clean ancillae; our local relay for these exhaustively checked S12/S6 templates; manwithacat with Codex for this frontier port. |
| Constrained λ_eff refit at fixed words, e1 released | HW-10, [Version 11](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/11) | — | Move W and H2_I on the e2/E_gap constraint surface with the integer words and compiled loaders frozen, letting e1 use its remaining headroom. Credit: our HW-03..HW-05 refit, applied to the newest public weights. |
| Weights and loaders combined across entries | HW-11, [Version 12](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/12) | HW-23 | Weights fit at fixed words are valid with any loaders for the same words, so the best weights and the cheapest loaders combine. Credit: parts assembly, with our HW-10 weights. |
| Systematic GF(2) kernel-relation search for QROM loaders | HW-12, [Version 13](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/13) | HW-13 | Add GF(2) relations that annihilate every high-monomial slice of a loader's mixed ANF tensor, keeping a perfectly paired rank-one basis; each admissible relation saves half a Toffoli per pass. Balanced round-robin choice over in-phase slot triples is what keeps pairing feasible. Credit: ours, on lightningv08's, Ahmed Berat Özer's, mrmeows' and matousakura's ideas, built with Lera's emitter. |
| Block-design relation plans with exchange repair | HW-13, [Version 14](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/14) | HW-14 | Partition the loader's slots into disjoint within-phase triples and draw a fixed number of relations from each triple's kernel; a basis-exchange repair then removes unpaired templates. Concentrating relations keeps each slot's constraint space small enough to pair perfectly (15 per triple: 35 − 2n partner directions). Credit: ours, extending HW-12. |
| Dead-word flips for QROM loaders | HW-14, [Version 15](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/15) | — | Angle words whose rotation acts on a zero-amplitude branch are exact don't-cares: changing them leaves every accuracy metric unchanged. Choosing their values to enlarge a loader's within-phase relation supply lets the planner find one more paired relation, saving a Toffoli per pass with no refit. Credit: ours, with the Rust qrom-plan search. |
| A reusable constrained refit tool (dfthc_refit) | HW-15, [Version 17](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/17) | HW-21 | The fixed-word constrained refit and column gauge of HW-03 to HW-10, packaged as one deterministic public tool: projected L-BFGS on log(λ_eff) with error_2 and E_gap held, error_1 as an inequality and near-zero h1′ eigenvalues held, alternated with the exact Jacobi column gauge, stopped when the KKT first-order gain and the realised gain both fall below 1e-5. Anyone can apply it to a new public family. Credit: ours (dfthc_refit 1.1.0); projected L-BFGS idea after anazemcev. |
| Re-planning a public loader's GF(2) relations from its own words | HW-16, [Version 19](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/19) | HW-17, HW-18 | A loader's Toffoli count is set by how many paired kernel relations its plan uses (HW-12 to HW-14). Starting from the published angle words, our relation planner finds a plan with four more products shared on each of the four plain loaders (0–3) of Ayodeji's new layout; Lera's unmodified emitter turns the plan into the circuit. Words, weights and accuracy do not move. Credit: ours (qrom-plan); emitter and table by Lera, layout by Ayodeji. |
| Loader re-planning as a repeatable step (dfthc_refit replan) | HW-17, [Version 20](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/20) | HW-23, HW-24 | The HW-16 re-planning packaged as a tool step: for each slot it recovers the loaded words, re-plans the loader with the relation planner, emits it with Lera's emitter and keeps it only if it has strictly fewer Toffolis and passes the native, width, layout and family checks; otherwise the shipped file is kept. Applied to a newer public family, it carries the HW-16 saving forward without redoing the work. Credit: ours (dfthc_refit 1.2.0-dev, local). |
| Padding rows as don't-cares in the loader search | HW-18, [Version 22](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/22) | HW-19, HW-23, HW-24 | Lera's loaders address 1,232 rows but the table has 1,220; the 12 padded rows are never read, so their values are free. Letting the relation search choose them lowers the rank of each loader's first output group by enough to save one product, with every loaded word unchanged. Credit: ours, on qwertykeypadapple's search. |
| Refit with the error margins held at the input's values | HW-21, [Version 24](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/24) | — | When a public table already sits at (or a hair past) the refit tool's default error caps, the refit is asked to hold error_1 and error_2 at the input's own values instead of tightening them first; the weights then move along the constraint surface and lower lambda_eff. Credit: manwithacat (James Barlow) with Claude (Anthropic), using our dfthc_refit tool. |
| Joining two public circuits where they are byte-identical | HW-22, [Version 24](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/24) | — | Two public circuits for the same task that agree byte for byte over a stretch can be cut there and joined: each side keeps the section where it is cheaper. The result uses no new gate, only the two authors' own operations. Credit: the two source circuits' authors (Akmal Xodarev; Alex Chilton and Leo Tang); the join by manwithacat with Claude (Anthropic). |
| Gauge-aware loader emission for a pi-gauge table | HW-23, [Version 25](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/25) | HW-24 | Lera's emitter borrows each output word's top bit as scratch and restores it from bit 6, which is right only when the table is stored with bit 7 = bit 6. The table of Heliosli's aba448aeadb5 (Syed Asad Ali's words in Atakan Aldemir's pi-gauge tree) flips the top bit on about 21% of entries by a pattern fixed per column and set by the low address bits (an exact symmetry: the rotations do not change), and its loaders are exact XOR oracles given the previous slot's flip pattern. Building the low-address cache first lets the emitter clear the borrowed bit of the incoming pattern and restore it with the outgoing one, so loaders planned one Toffoli cheaper can be emitted on this table; a word whose pattern changes between slots keeps its top bit (+1 qubit). Credit: manwithacat (James Barlow) with Claude (Anthropic), extending Lera's (marculera) tensorqrom emitter, in dfthc_refit 1.5.0-dev (local). |
| Chained loader re-planning on a newly published family | HW-24, Version 26 (this release) | — | A new public table is taken through the loader tools the day it appears: read its storage convention (here a pi gauge with 10 flip patterns, and brot = 8 read from the circuits because the weights file has no brot field), compare each loader with the planner's floor, re-plan only the loaders above it, and chain the runs, each starting from the previous run's output, with fresh seeds for the slots still above the floor. A loader changes only if it is strictly cheaper and passes a native check on its real inputs. Credit: manwithacat (James Barlow) with Claude (Anthropic), dfthc_refit 1.6.0-dev (local). |

## 4. All entries

One row per tweak, read from the dataset's `entries.csv`, which also holds the full verifier and draw results and every threshold, met or missed. ΔQ is the change in qubits and ΔT the change in mean Toffolis per shot. Expected GM is the change in the expected overall score, with the public frontier's circuits for the other two tasks, at the time of the entry.

In [ ]:
hw_csv = find("entries.csv")[0]
HW_DIR = hw_csv.parent
HW = list(csv.DictReader(open(hw_csv, newline="")))
show(pd.DataFrame([{
    "entry": e["id"], "date": e["date"], "task": e["task"], "change": e["title"],
    "base (author, circuit)": e["base_circuit"], "new circuit": e["candidate_sha12"],
    "ΔQ": int(e["delta_qubits"]), "ΔT per shot": f"{float(e['delta_toffoli_mean']):+,.1f}",
    "expected GM": f"{float(e['expected_gm_gain_pct']):+.3f}%",
    "failing draws (base → new)": (f"{e['draws_failing_base']} → {e['draws_failing_candidate']} of 48"
                                   if e["draws_failing_base"] else "no random draw"),
    "gates": ("missed: " + e["gates_missed"].split(":")[0]) if e["gates_missed"] else "all met",
    "base notebook": e["base_notebook"]} for e in HW]))
print(f"{len(HW)} entr{'y' if len(HW) == 1 else 'ies'} in {hw_csv.name} (sha256 {sha256(hw_csv)[:16]}...)")

In [ ]:
REQUIRED_ENTRIES = {'HW-24': 'ec58b2f5fdccef3d073eeb362322f95847868dbd65c684a44c36260fb1629528'}
mounted = {e['id']: e['candidate_sha256'] for e in HW}
assert all(mounted.get(id_) == sha for id_, sha in REQUIRED_ENTRIES.items()), \
    "the attached Hedge dataset is stale: refresh to the release's pinned version"

## 5. What did not work

Ideas that failed are kept with the same care as those that worked, in the [VQCC Negative Results](https://www.kaggle.com/datasets/manwithacat/vqcc-chemistry-negative-results) dataset: one row per idea, with its evidence tier, why it failed and what would reopen it. Before trying something, check there.

- **Porting our failure fix to yoyobar's circuit gains nothing (HW-01).** Our Quantum Voodoo failure fix rebuilt decoder groups 4–7 as homogeneous chains with their poles dropped. yoyobar's circuit already has groups 3–7 built that way, so there is no Toffoli saving left and no chance-zero failure to remove (measured per group, local engine).
- **Releasing more idle wires cannot reach 887 (HW-01).** Releasing q866 or the 7-wire block cannot lower the peak, because 34 stretches at 888 use every live wire (24-bit multiplications; census, structural).
- **A column gauge alone does not beat Isabe11aM00re's (HW-03).** From Özer's weights, exact Jacobi sweeps and a smoothed-L1 L-BFGS from the identity, from Isabe11aM00re's frame and from 128 random kicks found nothing below Isabe11aM00re's frame (local prescreen). On Özer's C43 polish a gauge alone gained only 0.00073 in λ_eff (HW-04 run).
- **Our earlier claim that the fixed-word refit sits at a constrained optimum (NR-0080) was start-specific.** From Heliosli's weights it stalled; from Özer's, Isabe11aM00re's and Lera's it keeps improving (HW-03, HW-04).
- **Jumping error_1 straight to its limit fails (HW-04).** The refit's Gauss–Newton correction could not move the error_1 target from 13.3 to 18 or 20 mHa in one go; raising it in steps works.
- **yoyobar's 892-qubit DQI circuit does not reach 891 by cheap means (HW-06).** CNOT-only freeing covers 60 of 216 full-width stretches; releasing and recomputing the held epoch flags would cost about 1,900 Toffolis; clearing allocator artefacts leaves 132 stretches at 892. The normalize step needs a redesign (static model; see the HW-06 section).
- **Nor by releasing the carried pole flag (HW-07).** It is the one wire idle across every 892-qubit stretch, but it holds [N_k = 0] ∧ P with P the parity of the error weight, a global bit that no wire carries where it would be recomputed; rebuilding it costs about 1,872 Toffolis per shot against about 905 that a qubit is worth (bit-parallel classical simulation on 4,096 valid syndromes). 891 needs a new inversion kernel.

## 6. Credits

- **HW-01** (dqi): base circuit `73e1b187b288` by **yoyobar** ([notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v8, Apache-2.0), on the decoder line of **SomeJoint (Agniv Banerjee)**; releasing a pole flag and recomputing it was first used publicly by **heliosli** on SomeJoint's decoder. Change: release-and-recompute (Quantum Voodoo line), the q852 release, the q878 reset and the handover erase, by manwithacat.
- **HW-03** (chemistry): weights by **Ahmed Berat Özer** (4eabe756bc36, [notebook](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), Apache-2.0); per-rank column-gauge idea and the base point ec11a966fb64 by **Isabe11aM00re** ([notebook](https://www.kaggle.com/code/isabe11am00re/qcc-v47-submission) v3, Apache-2.0); word table by **Atakan Aldemir**; the (8, 143, 60) factorization by **SomeJoint (Agniv Banerjee)**; loaders by **Heliosli, Dmitrii Gluzdov and Atakan Aldemir**. Change: the refit + gauge alternation and the gauge search, by manwithacat.
- **HW-04** (chemistry): C42 factor/word table and loaders by **Lera (marculera)** (e9a9289ab06e, [notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v6, Apache-2.0). Change: the e1-spending refit + gauge alternation, by manwithacat. (The method was first run on Ahmed Berat Özer's C43 polish e9eb9336a5ec.)
- **HW-05** (chemistry): C40 factorization, 8-bit open-pattern words and all 15 loaders by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9, Apache-2.0). Change: the fixed-word constrained W/H2_I refit by manwithacat (James Barlow), assisted by Codex (OpenAI). The refit method was developed in HW-03 and applied to C42 in HW-04.
- **HW-06** (dqi): base circuit `6d58bd6d0232` by **yoyobar** (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Change: measurement-based uncomputation at the 12 borrowed-wire flag ANDs, and the value tracker that found them, by manwithacat.
- **HW-07** (dqi): base circuit `c253111974fc`, our HW-06, on **yoyobar**'s `6d58bd6d0232` (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Reading Ali Haydar Özdağ's (ahoirg) public reproducible rebuild of SomeJoint's decoder sources ([notebook](https://www.kaggle.com/code/ahoirg/score-23-64254-reproducible-circuit-portfolio) v1) explained the normalize step's wires. Change: the two-wire measurement-based uncomputation of the 12 six-input flags, by manwithacat.
- **HW-08** (dqi): base circuit `b7ca9bce749d` by **SomeJoint (Agniv Banerjee)** ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi) v26, Apache-2.0): GPT-6 Astra Pro's q26 decoder round, with Claude putting SomeJoint's flag carry, which follows **Ali Haydar Özdağ**'s, on it; the flag carry is as in `bdee18489fac` (**yoyobar**'s one-carry history). Change: HW-06's and HW-07's measurement-based uncomputation at 24 sites, found by our census tool, by manwithacat.
- **HW-09** (dqi): Base by **Akmal Xodarev (Busya PRIME)**: [33 QCC V8 members](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) v9, from [Exact DQI Rewrites](https://www.kaggle.com/code/busyaprime/23-910-local-gm-exact-dqi-rewrites-892-qubits) v15, Apache-2.0. The core decoder is **SomeJoint (Agniv Banerjee)**'s q26 892-qubit design; busyaprime adds exact product caching, credits **Ahmed Berat Özer** for that idea, and incorporates our HW-07 ladder. Our S12/S6 templates were developed in the earlier local relay, following **Khattar and Gidney**'s conditionally clean ancilla method; this port and verification are by **manwithacat (James Barlow), assisted by Codex (OpenAI)**.
- **HW-10** (chemistry): C40 factorization and 8-bit words by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); angle loaders 0-3 and 14 by **Dmitrii Gluzdov** ([notebook](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source)), the rest Lera's; base weights by **yoyobar** ([notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v16), after refits by Isabe11aM00re, Heliosli, Ahmed Berat Özer and Alex Chilton; all Apache-2.0. Change: the constrained W/H2_I refit by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-11** (chemistry): C40 table and words by **Lera (marculera)**; angle loaders 0–3 recompiled by **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class) v8) with Lera’s public 5-word-QROM compiler, loaders 4–14 by **Dmitrii Gluzdov** and Lera; weights are our HW-10 refit of **yoyobar**’s weights (after refits by Isabe11aM00re, Heliosli, Ahmed Berat Özer and Alex Chilton); all Apache-2.0. Combination by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-12** (chemistry): C40 table, words and the 128-qubit loader emitter (`tensorqrom.build_circuit`) by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); the kernel-relation compiler ideas and quotient pairing (`kernel_planes.py`, `permuted_kernel.py`, `rank_parity.py`) from **lightningv08** (Vladislav Lavrentev), whose NOTICE credits **Ahmed Berat Özer** (generic kernel-relation emitter) and **mrmeows** (paired-product kernel, prefix cache, HMR emitter); the extended-relation idea and formula T_k = 89 + (525 − rel)/2 from **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class)); weights by **sigmaborov** ([notebook](https://www.kaggle.com/code/sigmaborov/22-85569-lb-exact-gauge-search) v15); all Apache-2.0. The relation search and the rebuilt loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-13** (chemistry): C40 table, words and the 128-qubit loader emitter (`tensorqrom.build_circuit`) by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); the kernel-relation compiler ideas and quotient pairing (`kernel_planes.py`, `permuted_kernel.py`, `rank_parity.py`) from **lightningv08** (Vladislav Lavrentev), whose NOTICE credits **Ahmed Berat Özer** (generic kernel-relation emitter) and **mrmeows** (paired-product kernel, prefix cache, HMR emitter); the extended-relation idea and formula T_k = 89 + (525 − rel)/2 from **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class)); weights by **SomeJoint** (Agniv Banerjee’s team; [dataset](https://www.kaggle.com/datasets/somejoint/qcc-ladder-parts) v1); all Apache-2.0. The relation plans (HW-12 search, block designs, exchange repair, layout search) and the rebuilt loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-14** (chemistry): C40 table, words and the 128-qubit loader emitter (`tensorqrom.build_circuit`) by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9); the kernel-relation compiler ideas and quotient pairing (`kernel_planes.py`, `permuted_kernel.py`, `rank_parity.py`) from **lightningv08** (Vladislav Lavrentev), whose NOTICE credits **Ahmed Berat Özer** (generic kernel-relation emitter) and **mrmeows** (paired-product kernel, prefix cache, HMR emitter); the extended-relation idea and formula T_k = 89 + (525 − rel)/2 from **matousakura** ([notebook](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class)); weights by **Heliosli** ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) v42); all Apache-2.0. The dead-word flip search (Rust qrom-plan screen/stack/batch), relation plans and the rebuilt loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-15** (chemistry): Angle loaders and starting weights by **Agniv Banerjee (SomeJoint)** ([qcc-ladder-parts dataset](https://www.kaggle.com/datasets/somejoint/qcc-ladder-parts) v4, family `4a9adf368a10`); the (9, 143, 39) C = 39 table by **Syed Asad Ali** ([notebook](https://www.kaggle.com/code/syed3000/vqcc-24-43-portfolio) v1, `b9b05e14dbdb`), built on **Lera (marculera)**'s DF-THC compact-groups pipeline ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian)); all Apache-2.0. The weight refit by manwithacat (James Barlow) with our public [dfthc_refit](https://www.kaggle.com/code/manwithacat/dfthc-refit) 1.1.0 tool, assisted by Claude (Anthropic); its projected L-BFGS step follows an idea from anazemcev's public Apache-2.0 refit.
- **HW-16** (chemistry): Angle loaders 4–14 by **Ayodeji** ([notebook](https://www.kaggle.com/code/ayodejiibrahimlateef/vqcc-24-56308-local) v4, family `0ed964465654`) and weights by **Agniv Banerjee (SomeJoint)** ([qcc-ladder-parts dataset](https://www.kaggle.com/datasets/somejoint/qcc-ladder-parts) v8, family `a56ec354e8b8`), on the (8, 143, 45) table and emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `4ae679db726d`); all Apache-2.0. Loaders 0–3 re-planned by manwithacat (James Barlow) with our qrom-plan relation planner, assisted by Claude (Anthropic), and emitted with Lera's unmodified emitter.
- **HW-17** (chemistry): Angle loaders 4–14 by **Ayodeji** ([notebook](https://www.kaggle.com/code/ayodejiibrahimlateef/vqcc-24-56308-local) v4, family `0ed964465654`) with loader 10 and the weights by **Isabe11aM00re** ([notebook](https://www.kaggle.com/code/isabe11am00re/qcc-v66s33w-submission) v2, family `35694008ca5e`), on the (8, 143, 45) table and emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `4ae679db726d`); all Apache-2.0. Loaders 0–3 re-planned by manwithacat (James Barlow) with our qrom-plan relation planner (as HW-16), assisted by Claude (Anthropic), and emitted with Lera's unmodified emitter.
- **HW-18** (chemistry): Angle-loader search code (`kernel_search.py`, run unmodified) and the shipped loaders of family `a344f422f2e0` by **qwertykeypadapple** ([notebook](https://www.kaggle.com/code/qwertykeypadapple/qcc-kernel-search-24-65) v1 and its dataset, Apache-2.0); weights by **Agniv Banerjee** (family `a56ec354e8b8`, Apache-2.0); the (8, 143, 45) table and the emitter by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, Apache-2.0). Padded-row choice and re-emission of all 15 loaders by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-19** (chemistry): Word table, angle loaders 0-3 and weights of family `f4291619f2fd` by **Syed Asad Ali** ([dataset](https://www.kaggle.com/datasets/syed3000/vqcc-own-circuit-portfolio) v3 and [notebook](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) v3, Apache-2.0; their own fit and rounding of an (8, 143, 45) table in Lera's layout); angle-loader search code (`kernel_search.py`, run unmodified) by **qwertykeypadapple** ([notebook](https://www.kaggle.com/code/qwertykeypadapple/qcc-kernel-search-24-65) v1 and its dataset, Apache-2.0); the emitter by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, Apache-2.0). Padded-row re-planning of loaders 4-14 by manwithacat (James Barlow), assisted by Claude (Anthropic), as in HW-18.
- **HW-21** (chemistry): Base family `aba448aeadb5` by **Heliosli** ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) v10, Apache-2.0): Heliosli's weight refinement of **Akmal Xodarev (busyaprime)**'s refit, on **Syed Asad Ali**'s (8, 143, 45) word table with **Atakan Aldemir**'s pi-gauge tree and angle loaders, integrated by **Kirill Tymchishin** (as credited in Heliosli's notebook). Words and all 15 loaders are theirs, unchanged. Weight refit (three chained runs of our `dfthc_refit` tool, error margins held) by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-22** (dqi): Unsynthesis section from `fd0ef09efd45` by **Akmal Xodarev (busyaprime)** ([33 QCC V8 members](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) v10, Apache-2.0), their exact rewrite of **Syed Asad Ali**'s 891-qubit decoder `fc3ba81614e1`. Everything else, including the unextract section, from `1c43b7ff2fcf`: **Alex Chilton**'s decoder version, released by **Leo Tang** in [VQCC T7421 Submission](https://www.kaggle.com/code/leotang09/vqcc-t7421-submission) v17 (Apache-2.0), on the public 891-qubit decoder line (Syed Asad Ali, Agniv Banerjee and others, as credited there). The join of the two public circuits and its verification are by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-23** (chemistry): Base: our HW-21 (dataset v23 / notebook v24), a weight refit of Heliosli's `aba448aeadb5` ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) v10, Apache-2.0). That family is **Heliosli**'s weight refinement of **Akmal Xodarev (busyaprime)**'s refit, on **Syed Asad Ali**'s (8, 143, 45) word table with **Atakan Aldemir**'s pi-gauge tree and angle loaders, integrated by **Kirill Tymchishin** (as credited in Heliosli's notebook). Words and 11 of the 15 loaders are theirs, unchanged; the weights are HW-21's. The four new loaders keep the planning model and emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `tensorqrom`, Apache-2.0); the shared planner's construction follows **i want to run gemma 9b**'s coded loaders and Syed Asad Ali's block partition, with pairing after **lightningv08**. Plans and gauge-aware emission (our `dfthc_refit` 1.5.0-dev) by manwithacat (James Barlow), assisted by Claude (Anthropic).
- **HW-24** (chemistry): Base: **Syed Asad Ali**'s public chemistry family `eff6a4863f08` ([notebook](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) v4, Apache-2.0): their own (8, 143, 44) word table, their own loader compiler and their own weights (they credit Heliosli for the refit up to the accuracy gate). The table, the weights (`submission.npz`) and loaders 0–3 are theirs, unchanged. The eleven new loaders (slots 4–14) come from our `dfthc_refit` shared planner, whose construction follows **i want to run gemma 9b**'s coded loaders, with Syed Asad Ali's block partition and pairing after **lightningv08**, and are emitted with a gauge-aware version of the emitter of **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v10, `tensorqrom`, Apache-2.0; Lera's notice credits mrmeows and Heliosli), which handles the pi-gauge storage convention introduced by **Atakan Aldemir**. Plans and emission (our `dfthc_refit` 1.6.0-dev) by manwithacat (James Barlow), assisted by Claude (Anthropic).

The circuits are files of operations derived from other people's Apache-2.0 work; the dataset's NOTICE lists it. The tweaks are by manwithacat (James Barlow), made with substantial help from Claude (Anthropic) and Codex (OpenAI) under his direction; he chose the questions, approved each experiment and checked the claims.

## 7. How to reproduce

1. **Get the circuits.** Attach the [VQCC Hedge Wizardry dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry). `entries.csv` gives each circuit's SHA-256 (`SHA256SUMS` lists every file; a chemistry entry's hash is its family hash). The base circuits are in the [VQCC public circuits dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits) under `circuits/<task>/<sha12>`.
2. **Verify on Kaggle.** Copy this notebook and run it (CPU, internet off). It installs our [kickmix fork](https://github.com/manwithacat/kickmix-macos) offline from the attached wheel and runs the unchanged official verifiers on the portfolio. The entry gate below checks every entry's cost against `entries.csv`. The verifier seeds every shot from the file's SHA-256, so the same bytes give the same cost at the default salt.
3. **Verify locally.** With kickmix built from our fork (macOS) or the official Linux build:
   - **HW-01**: run the official `dqi` verifier on `circuits/dqi/5faba4591853.kmb` with an empty salt: PASS, 888 qubits, cost 401,986,129,571.42.
   - **HW-03**: run the official `dfthcsos` verifier on `circuits/chemistry/e148cacdd4ad/` with an empty salt: PASS, 430 qubits, cost 337,651,009.06.
   - **HW-04**: run the official `dfthcsos` verifier on `circuits/chemistry/02f8f7d9d75a/` with an empty salt: PASS, 428 qubits, cost 331,025,488.43.
   - **HW-05**: run the official `dfthcsos` verifier on `circuits/chemistry/ff2b2733c1fc/` with an empty salt: PASS, 428 qubits, cost 326,959,774.66.
   - **HW-06**: run the official `dqi` verifier on `circuits/dqi/c253111974fc.kmb` with an empty salt: PASS, 892 qubits, cost 382,537,501,561.76.
   - **HW-07**: run the official `dqi` verifier on `circuits/dqi/1779b4e5967f.kmb` with an empty salt: PASS, 892 qubits, cost 384,137,374,413.52.
   - **HW-08**: run the official `dqi` verifier on `circuits/dqi/5e86f69a9611.kmb` with an empty salt: PASS, 892 qubits, cost 383,320,859,120.03.
   - **HW-09**: run the official `dqi` verifier on `circuits/dqi/1698e864a3a6.kmb` with an empty salt: PASS, 892 qubits, cost 378,829,899,786.41.
   - **HW-10**: run the official `dfthcsos` verifier on `circuits/chemistry/7377be20cff0/` with an empty salt: PASS, 428 qubits, cost 326,764,450.98.
   - **HW-11**: run the official `dfthcsos` verifier on `circuits/chemistry/cc079d42a714/` with an empty salt: PASS, 428 qubits, cost 324,791,746.23.
   - **HW-12**: run the official `dfthcsos` verifier on `circuits/chemistry/b94b469b148a/` with an empty salt: PASS, 428 qubits, cost 319,274,986.45.
   - **HW-13**: run the official `dfthcsos` verifier on `circuits/chemistry/3aac21436c28/` with an empty salt: PASS, 428 qubits, cost 317,777,218.02.
   - **HW-14**: run the official `dfthcsos` verifier on `circuits/chemistry/7d372981d3ce/` with an empty salt: PASS, 428 qubits, cost 317,574,154.53.
   - **HW-15**: run the official `dfthcsos` verifier on `circuits/chemistry/bf016fb054d5/` with an empty salt: PASS, 428 qubits, cost 313,664,330.04.
   - **HW-16**: run the official `dfthcsos` verifier on `circuits/chemistry/c6d391fdc236/` with an empty salt: PASS, 427 qubits, cost 310,821,125.36.
   - **HW-17**: run the official `dfthcsos` verifier on `circuits/chemistry/8760de16eb95/` with an empty salt: PASS, 427 qubits, cost 310,780,568.95.
   - **HW-18**: run the official `dfthcsos` verifier on `circuits/chemistry/160fc1ecc073/` with an empty salt: PASS, 427 qubits, cost 308,604,860.13.
   - **HW-19**: run the official `dfthcsos` verifier on `circuits/chemistry/5619c37ef6bd/` with an empty salt: PASS, 427 qubits, cost 306,723,130.32.
   - **HW-21**: run the official `dfthcsos` verifier on `circuits/chemistry/9706e2c7ecff/` with an empty salt: PASS, 427 qubits, cost 303,854,940.78.
   - **HW-22**: run the official `dqi` verifier on `circuits/dqi/b99bf23306f6.kmb` with an empty salt: PASS, 891 qubits, cost 375,320,367,805.64.
   - **HW-23**: run the official `dfthcsos` verifier on `circuits/chemistry/81db735f32b5/` with an empty salt: PASS, 427 qubits, cost 303,694,816.97.
   - **HW-24**: run the official `dfthcsos` verifier on `circuits/chemistry/ec58b2f5fdcc/` with an empty salt: PASS, 427 qubits, cost 301,875,698.74.
4. **DQI draws.** The 48 draws come from our local engine with fixed seeds shared by base and candidate. That engine is not published, so these numbers cannot be re-run from public files. The per-draw costs and errors are in `evidence/<entry>/draws_48.csv`.
5. **Read the change.** A `.kmb` circuit is a list of operations. The fork's circuit viewer (`tools/circuit-viewer/render.py`, including `ledger --compare`) shows where base and candidate differ. For chemistry entries, `evidence/<entry>/metrics.json` and `alternation_curve.csv` give the metrics of every stage.

## The submission this notebook makes

The hidden cells below:

1. pick the cheapest public circuit per task from the public dataset's manifest (chemistry: a verified assembly of public parts if that is cheaper than the best single family);
2. for each task, swap in the cheapest Hedge Wizardry entry if it is cheaper than the public pick, after checking its SHA-256;
3. install our kickmix fork offline;
4. run the unmodified official verifiers at full settings;
5. verify this release's new entries and match their cost to `entries.csv` (the entry gate; earlier entries are hash-checked);
6. write `submission.zip` only if everything passes, with `check.json`, `entry_gates.json` and `receipt.json`.

In [ ]:
manifest = find("manifest.csv")[0]
DATA = manifest.parent
rows = list(csv.DictReader(open(manifest, newline="")))
print(f"manifest: {manifest} ({len(rows)} circuits, sha256 {sha256(manifest)[:16]}...)")

picks = {}
for task in REFERENCE:
    scored = rank_order([r for r in rows if r["task"] == task])  # tie group first, original first
    picks[task] = scored[0]
    best_cost = min(rank_cost(r) for r in scored)
    ties = [r["sha12"] for r in scored[1:] if rank_cost(r) <= best_cost * (1 + REL_TIE)]
    print(f"{task:9} {len(scored):2} candidates; picked {scored[0]['sha12']} "
          f"ranking cost {rank_cost(scored[0]):.6e} ({rank_basis(scored[0])}); "
          f"default-salt cost {float(scored[0]['local_task_score']):.6e}"
          + (f"; ties within noise, earlier public kept: {', '.join(ties)}" if ties else ""))

# Chemistry: a verified assembly of public parts replaces the best single family only if it is cheaper
ASSEMBLY = None
best_json = DATA / "parts" / "assembly_best.json"
if best_json.is_file():
    a = json.loads(best_json.read_text())
    single = float(picks["chemistry"]["local_task_score"])
    print(f"parts/assembly_best.json: verified {a['verified_cost']:.6e} ({a['verify_date']}) "
          f"vs best single family {picks['chemistry']['sha12']} {single:.6e}")
    if a.get("verified_status") == "PASS" and float(a["verified_cost"]) < single:
        ASSEMBLY = a

shutil.rmtree(SUB, ignore_errors=True)
SUB.mkdir(parents=True)
for task, r in picks.items():
    if task == "chemistry" and ASSEMBLY:
        fam = SUB / "dfthcsos"
        fam.mkdir()
        files = [(ASSEMBLY["npz"]["dataset_path"], ASSEMBLY["npz"]["sha256"], "submission.npz")]
        files += [(s["dataset_path"], s["file_sha256"], f"angle_k_{s['slot']}.kmb") for s in ASSEMBLY["slots"]]
        for path, digest, name in files:
            assert sha256(DATA / path) == digest, f"{path} does not match parts/assembly_best.json"
            shutil.copyfile(DATA / path, fam / name)
        authors = sorted({s["author"] for s in ASSEMBLY["slots"]})
        picks["chemistry"] = {
            "task": "chemistry", "sha12": "assembled", "sha256": family_sha256(fam),
            "qubits": ASSEMBLY["qubits"], "local_task_score": str(ASSEMBLY["verified_cost"]),
            "author": "assembled: loaders " + ", ".join(authors) + "; weights " + ASSEMBLY["npz"]["author"],
            "notebook_title": "assembly of public chemistry parts (parts/assembly_best.json)",
            # link the base notebook (the weights' source family), as the headline table does; the
            # parts and their authors are listed in the chemistry section
            "source_notebook_url": ASSEMBLY["npz"].get("notebook") or
                                   "https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits",
            "source_notebook_version": ASSEMBLY["npz"].get("notebook_version", ""),
            "release_date": ASSEMBLY["verify_date"],
            "core_author": ASSEMBLY["word_table_origin"]["author"],
            "change_summary": f"cheapest public circuit per angle loader with the best weights on word table "
                              f"{ASSEMBLY['table_key']}", "also_found_in": ""}
        print(f"chemistry: assembled from {len(files)} dataset files, every SHA-256 checked")
    elif task == "chemistry":
        src = DATA / "circuits" / "chemistry" / r["sha12"]
        assert family_sha256(src) == r["sha256"], f"chemistry family {r['sha12']} does not match the manifest"
        shutil.copytree(src, SUB / "dfthcsos")
    else:
        src = DATA / "circuits" / task / f"{r['sha12']}.kmb"
        assert sha256(src) == r["sha256"], f"{task} {r['sha12']} does not match the manifest"
        shutil.copyfile(src, SUB / f"{task}.kmb")
print("assembled and hash-checked:", sorted(p.name for p in SUB.iterdir()))

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame([{"task": t, "candidates": sum(1 for x in rows if x["task"] == t and x["local_task_score"]),
                       "picked": r["sha12"], "qubits": r.get("qubits"),
                       "ranking cost": rank_cost(r), "ranked by": rank_basis(r),
                       "local cost (default salt)": float(r["local_task_score"]), "author": r["author"],
                       "sha256 checked": True} for t, r in picks.items()]).set_index("task"))

In [ ]:
# Hedge Wizardry entries: verify each file's SHA-256, then let the cheapest entry per task replace the public
# pick only if it beats it on the same ranking cost (DQI: the 48-draw mean when the entry has one, against the
# public pick's rank_cost; others: the default-salt cost) by more than the tie threshold REL_TIE.
HW_FILES = {}
for e in HW:
    src = HW_DIR / e["dataset_path"]
    got = family_sha256(src) if e["task"] == "chemistry" else sha256(src)
    assert got == e["candidate_sha256"], f"{e['id']}: {src.name} does not match entries.csv"
    HW_FILES[e["id"]] = src
HW_USED = {}
for task in REFERENCE:
    def entry_cost(e):
        return float(e.get("draws_mean_cost_candidate") or e["local_task_score"]) if task == "dqi"                         else float(e["local_task_score"])
    mine = sorted((e for e in HW if e["task"] == task), key=entry_cost)
    if not mine:
        continue
    e, public = mine[0], picks[task]
    pub_cost = rank_cost(public)
    if not entry_cost(e) < pub_cost * (1 - REL_TIE):
        print(f"{task}: public {public['sha12']} ({pub_cost:.6e}, {rank_basis(public)}) stays; best entry "
              f"{e['id']} {e['candidate_sha12']} ({entry_cost(e):.6e}) does not beat it beyond the tie threshold")
        continue
    src = HW_FILES[e["id"]]
    if task == "chemistry":
        shutil.rmtree(SUB / "dfthcsos", ignore_errors=True)
        shutil.copytree(src, SUB / "dfthcsos")
        ASSEMBLY = None
    else:
        shutil.copyfile(src, SUB / f"{task}.kmb")
    picks[task] = {
        "task": task, "sha12": e["candidate_sha12"], "sha256": e["candidate_sha256"], "qubits": e["qubits"],
        "local_task_score": e["local_task_score"],
        "author": f"manwithacat (Hedge Wizardry {e['id']}) on {e['base_author']}'s {e['base_sha256'][:12]}",
        "notebook_title": f"VQCC Hedge Wizardry {e['id']}: {e['title']}",
        "source_notebook_url": "https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook",
        "source_notebook_version": "", "release_date": e["date"], "core_author": e["base_author"],
        "change_summary": e["card_summary"], "also_found_in": ""}
    HW_USED[task] = e["id"]
    print(f"{task}: Hedge Wizardry {e['id']} {e['candidate_sha12']} ({e['qubits']} qubits, local cost "
          f"{float(e['local_task_score']):.6e}) replaces public {public['sha12']} by {public['author']} "
          f"({pub_cost:.6e}, {rank_basis(public)}): {entry_cost(e) / pub_cost - 1:+.3%}")

In [ ]:
tool_dirs = [Path(p).parent for root in ("/kaggle/usr/lib", "/kaggle/input")
             for p in sorted(glob.glob(f"{root}/**/kickmix_tools.py", recursive=True))]
assert tool_dirs, "attach the kickmix_tools utility script"
sys.path.insert(0, str(tool_dirs[0]))
import kickmix_tools as kt
print("kickmix_tools", kt.__version__, "from", tool_dirs[0])

wheel_dir = find(WHEEL_GLOB)[0].parent
FORK_DIR = kt.install(str(wheel_dir), target="/tmp/kickmix_fork")
FORK_PROVENANCE = kt.provenance()
print("fork provenance:", FORK_PROVENANCE)
ENGINE_DIR = kt.install_engine()  # wheel from kickmix-engine-wheels, checked against its SHA256SUMS
print("kickmix-engine:", ENGINE_DIR)
print("competition data:", kt.competition_dir())

In [ ]:
# Verification reuse: a pick whose exact bytes already have an official-verifier PASS is not re-verified here; the
# receipt says which report each task relies on. Picks verified here run on kickmix-engine (all three tasks since
# kickmix-tools v5, 2026-11-03). FORCE_FULL_VERIFY = True (builder: --force-verify) skips reuse
# and runs the unchanged official verifier on the fork for all three.
FORCE_FULL_VERIFY = False
# REUSE_VERIFIED = False (builder: --no-reuse) verifies every pick this run on the default builds (engine timing runs)
REUSE_VERIFIED = True
ENGINE_TASKS = ("invsqrt", "dqi", "chemistry")

def verify_build(task):
    return "fork" if FORCE_FULL_VERIFY or task not in ENGINE_TASKS else "engine"

BUILD_LABEL = {"engine": "kickmix-engine (official verifier file, engine simulator)",
               "fork": "official verifier (kickmix fork)"}

def _num(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if v == v else None

def reuse_record(task, pick, hedge_selected=None, hedge_entries=(), assembly=None):
    # The earlier official-verifier PASS for these exact bytes, or None. Sources: the Hedge Wizardry entry that
    # replaced the pick (its recorded verifier gate, candidate SHA-256 matched); the chemistry parts assembly
    # (parts/assembly_best.json, verified locally; every file hash-checked when assembled); else the manifest row's
    # local official-verifier report (local_report_sha256; the file's SHA-256 was checked when it was copied).
    hw = (hedge_selected or {}).get(task)
    if hw:
        e = next((x for x in hedge_entries if x.get("id") == hw["entry"]), None)
        text = " ".join(((e or {}).get("verifier_result") or "").split())
        if e and e.get("candidate_sha256") == pick["sha256"] and "verifier PASS" in text \
                and _num(e.get("local_task_score")):
            return {"source": f"Hedge Wizardry {e['id']} entry gate", "date": e.get("date", ""),
                    "report": text[:140], "score": _num(e["local_task_score"]), "qubits": _num(e.get("qubits")),
                    "toffoli_count": _num(e.get("toffoli_mean")), "metrics": None}
        return None
    if task == "chemistry" and assembly:
        if assembly.get("verified_status") == "PASS" and assembly.get("verifier_report_sha256") \
                and _num(assembly.get("verified_cost")):
            return {"source": "parts/assembly_best.json (local official verifier, default salt)",
                    "date": assembly.get("verify_date", ""),
                    "report": "report sha256 " + assembly["verifier_report_sha256"][:16] + "…",
                    "score": _num(assembly["verified_cost"]), "qubits": _num(assembly.get("qubits")),
                    "toffoli_count": _num(assembly.get("toffoli")), "metrics": None}
        return None
    tier = pick.get("local_score_tier") or ""
    if pick.get("local_report_sha256") and tier.startswith("local official verifier") \
            and _num(pick.get("local_task_score")):
        metrics = None
        if task == "chemistry" and all(_num(pick.get(k)) is not None for k in ("error_1_mHa", "error_2_mHa", "E_gap")):
            metrics = {"error_1": _num(pick["error_1_mHa"]) / 1000, "error_2": _num(pick["error_2_mHa"]) / 1000,
                       "E_gap": _num(pick["E_gap"])}
        return {"source": f"manifest.csv local report ({tier})", "date": pick.get("first_seen", ""),
                "report": "report sha256 " + pick["local_report_sha256"][:16] + "…",
                "score": _num(pick["local_task_score"]), "qubits": _num(pick.get("qubits")),
                "toffoli_count": _num(pick.get("T_total")) if task == "chemistry" else None, "metrics": metrics}
    return None

def reused_result(rec):
    # A result shaped like kt.verify()'s, from a reuse record (fields the record lacks stay None: shown as —)
    res = {"status": "PASS", "score": rec["score"], "num_qubits": int(rec["qubits"]) if rec.get("qubits") else None,
           "toffoli_count": rec.get("toffoli_count")}
    if rec.get("metrics"):
        res["metrics"] = rec["metrics"]
    return {"result": res, "wall_seconds": None, "peak_pss_sum_mb": None, "peak_rss_sum_mb": None, "reused": rec}


PATHS = {"invsqrt": SUB / "invsqrt.kmb", "dqi": SUB / "dqi.kmb", "chemistry": SUB / "dfthcsos"}
results, check_rows, VERIFICATION = {}, [], {}
for task, circuit in PATHS.items():
    rec = None if FORCE_FULL_VERIFY or not REUSE_VERIFIED else reuse_record(
        task, picks[task], globals().get("HEDGE_SELECTED"), globals().get("HW", ()), ASSEMBLY)
    build = verify_build(task)
    r = reused_result(rec) if rec else dict(kt.verify(VERIFIER_TASK[task], circuit, build=build), reused=None)
    VERIFICATION[task] = (f"reused: {rec['source']}, {rec['date']}; {rec['report']}" if rec
                          else f"verified this run: {BUILD_LABEL[build]}")
    results[task] = r
    res = r["result"]
    listed = float(picks[task]["local_task_score"])
    check_rows.append({
        "task": task, "circuit": picks[task]["sha12"], "status": res["status"],
        "verification": "reused" if rec else f"this run: {BUILD_LABEL[build]}",
        "cost": res["score"], "source cost": listed, "cost / source": res["score"] / listed,
        "times cheaper than reference": REFERENCE[task] / res["score"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": res.get("toffoli_count"),
        "value errors": res.get("value_errors") if task == "dqi" else None,
        "phase errors": res.get("phase_errors") if task == "dqi" else None,
        "wall s": None if rec else round(r["wall_seconds"], 1),
        "peak MB": None if rec else round(r["peak_pss_sum_mb"] or r["peak_rss_sum_mb"] or 0)})
checks = pd.DataFrame(check_rows).set_index("task")
FORMATS = {"cost": "{:.6e}", "source cost": "{:.6e}", "cost / source": "{:.9f}",
           "times cheaper than reference": "{:.4f}", "qubits": "{:.0f}", "Toffolis (mean)": "{:,.1f}",
           "value errors": "{:.0f}", "phase errors": "{:.0f}", "wall s": "{:.1f}", "peak MB": "{:,.0f}"}
shown = checks.copy()
for col, fmt in FORMATS.items():  # value/phase errors exist for DQI only; blank cells show as —
    shown[col] = [fmt.format(v) if pd.notna(v) else "—" for v in checks[col]]
display(shown)
print("— = not applicable (only the DQI verifier counts value and phase errors) or not recorded in a reused report.")
n_reused = sum(1 for r in results.values() if r.get("reused"))
print(f"verification: {len(results) - n_reused} pick(s) verified this run, {n_reused} reused an earlier "
      f"official-verifier PASS for the exact bytes (FORCE_FULL_VERIFY = {FORCE_FULL_VERIFY})")
for t, v in VERIFICATION.items():
    print(f"  {t}: {v}")
failed = [t for t, r in results.items() if r["result"]["status"] != "PASS"]
assert not failed, f"not writing a submission: {failed} did not pass"

ratios = {t: REFERENCE[t] / results[t]["result"]["score"] for t in REFERENCE}
gm = prod(ratios.values()) ** (1 / 3)
# DQI cost depends on the draw. The headline uses the DQI ranking cost (mean over a 48-draw sweep, or the
# zero-failure cost); gm is this run's single default-salt draw (2026-10-31: the grader uses its own hidden draw).
ratios_exp = dict(ratios, dqi=REFERENCE["dqi"] / rank_cost(picks["dqi"]))
gm_exp = prod(ratios_exp.values()) ** (1 / 3)
zf = picks['dqi'].get('dqi_zero_failure_cost')  # best case: no failed DQI shots in a draw
gm_zero = prod(dict(ratios, dqi=REFERENCE['dqi'] / float(zf)).values()) ** (1 / 3) if zf else None
# Error bounds (2026-11-01). Only DQI's cost depends on the draw: cost = R(k) x Q^2 x Toffolis, k the failed
# shots of the draw. The pick's 48-draw expected cost over its zero-failure cost is E[R(k)] / R(0); with k
# Poisson(lam) that fixes lam, so every file with a 48-draw sweep gets a measured final-draw risk.
R_K = [1.19126, 1.19673, 1.20139, 1.20558, 1.20946, 1.21311, 1.21657]  # R(k), 102,400 shots (verifier)
def _pois(lam):
    from math import exp, factorial
    p = [exp(-lam) * lam ** k / factorial(k) for k in range(len(R_K))]
    p[-1] += 1 - sum(p)
    return p
DQI_RISK = None
ec = picks['dqi'].get('dqi_expected_cost')
if zf and ec and float(ec) >= float(zf):
    target = float(ec) / float(zf)
    lo, hi = 0.0, 10.0
    for _ in range(80):  # E[R(k)]/R(0) rises with lam: bisect
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if sum(q * r / R_K[0] for q, r in zip(_pois(mid), R_K)) < target else (lo, mid)
    lam = (lo + hi) / 2
    pk = _pois(lam)
    gm_k = lambda k: prod(dict(ratios, dqi=REFERENCE['dqi'] / (float(zf) * R_K[k] / R_K[0])).values()) ** (1 / 3)
    DQI_RISK = {"failures_per_draw": lam, "p_none": pk[0], "p_one_or_more": 1 - pk[0],
                "p_two_or_more": 1 - pk[0] - pk[1], "gm_lucky": gm_k(0), "gm_one": gm_k(1), "gm_two": gm_k(2),
                "basis": "Poisson fit to the 48-draw expected cost / zero-failure cost (manifest)"}
print(f"expected geometric mean: {gm_exp:.5f} (DQI {rank_basis(picks['dqi'])}); "
      f"this run's single default-salt draw: {gm:.5f}")
checks.to_csv(WORK / "checks.csv")

### Entry gate

Every entry's circuit, verified here on Kaggle with our [kickmix-engine](https://github.com/manwithacat/kickmix-engine), with circuits loaded by our [kickmix fork](https://github.com/manwithacat/kickmix-macos). Both run the competition's own verifier code, and together they reproduce the official verifier's results field for field (status, cost and every metric, checked on PASS and FAIL controls); setting `FORCE_FULL_VERIFY = True` runs the unchanged official verifier instead. Both are open source and install offline from public Kaggle datasets via `kickmix_tools`, so anyone can use the same process. An entry already in the portfolio reuses the run above, and this release's new entries are verified separately. Earlier entries passed this gate in the release where they were the headline; their file hashes are checked again, and `VERIFY_ALL_ENTRIES = True` re-runs them all. A verified entry's cost must match the local record in `entries.csv` to 1e-9, which holds because the verifier seeds every shot from the file's SHA-256.

In [ ]:
# The new entries of this release and any entry in the portfolio are verified here. An earlier entry passed
# this same gate in the release where it was the headline; its file SHA-256 was checked again above, so it
# is not re-run unless VERIFY_ALL_ENTRIES = True (each earlier entry costs about 20-35 s).
VERIFY_ALL_ENTRIES = False
ENTRY_GATES = []
for e in HW:
    task = e["task"]
    listed = float(e["local_task_score"])
    if picks[task]["sha256"] == e["candidate_sha256"]:
        res, how = results[task]["result"], "portfolio run"
    elif VERIFY_ALL_ENTRIES or e["id"] in REQUIRED_ENTRIES:
        _b = verify_build(task)
        res, how = (kt.verify(VERIFIER_TASK[task], HW_FILES[e["id"]], build=_b)["result"],
                    f"separate run: {BUILD_LABEL[_b]}")
    else:
        ENTRY_GATES.append({
            "entry": e["id"], "circuit": e["candidate_sha12"],
            "verifier run": "not re-run: SHA-256 matches; passed this gate in an earlier release",
            "status": "PASS", "qubits": e.get("qubits"), "Toffolis (mean)": "—", "cost (Kaggle)": "—",
            "entries.csv cost": f"{listed:.6e}", "matches to 1e-9": True,
            "in this submission": HW_USED.get(task) == e["id"]})
        continue
    rel = abs(res["score"] / listed - 1)
    ENTRY_GATES.append({
        "entry": e["id"], "circuit": e["candidate_sha12"], "verifier run": how, "status": res["status"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": f"{res.get('toffoli_count', float('nan')):,.1f}",
        "cost (Kaggle)": f"{res['score']:.6e}", "entries.csv cost": f"{listed:.6e}",
        "matches to 1e-9": rel <= 1e-9, "in this submission": HW_USED.get(task) == e["id"]})
show(pd.DataFrame(ENTRY_GATES))
assert all(g["status"] == "PASS" for g in ENTRY_GATES), "an entry failed verification"
assert all(g["matches to 1e-9"] for g in ENTRY_GATES), "a hosted entry cost differs from its local record"
(WORK / "entry_gates.json").write_text(json.dumps(ENTRY_GATES, indent=1, default=str))

In [ ]:
members = [("invsqrt.kmb", SUB / "invsqrt.kmb"), ("dqi.kmb", SUB / "dqi.kmb")]
members += [(f"dfthcsos/{p.relative_to(SUB / 'dfthcsos').as_posix()}", p)
            for p in sorted((SUB / "dfthcsos").rglob("*")) if p.is_file()]
with zipfile.ZipFile(WORK / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, path in members:
        z.write(path, name)
(WORK / "check.json").write_text(json.dumps({
    "manifest_sha256": sha256(manifest), "submission_sha256": sha256(WORK / "submission.zip"),
    "geometric_mean_local": gm, "fork": FORK_PROVENANCE,
    "picks": {t: {k: picks[t][k] for k in ("sha256", "author", "notebook_title", "source_notebook_url",
                                            "release_date", "local_task_score")} for t in picks},
    "hedge_blend": globals().get("HEDGE_SELECTED", {}),
    "chemistry_assembly": ASSEMBLY, "verification": VERIFICATION,
    "results": {t: r["result"] for t, r in results.items()}}, indent=1, default=str))
shutil.rmtree(SUB)
print("wrote", WORK / "submission.zip", "and", WORK / "check.json")

### Verification receipt

The selected circuits, their sources, file hashes and verifier results, followed by checks that the costs reproduce their records and the run completes within two hours.

In [ ]:
WALL = time.monotonic() - T0
receipt = pd.DataFrame([{
    "task": TASK_NAMES[t], "circuit": picks[t]["sha12"], "sha256": picks[t]["sha256"][:16] + "…",
    "published by": picks[t]["author"], "version": picks[t]["source_notebook_version"],
    "verdict": results[t]["result"]["status"], "verification": VERIFICATION[t],
    "times cheaper than reference": REFERENCE[t] / results[t]["result"]["score"],
    "link": picks[t]["source_notebook_url"]} for t in REFERENCE])
show(receipt)
CHEM_PARTS = None
if ASSEMBLY:
    print("Chemistry was assembled from public parts: each angle loader, the weights and the word table's origin")
    CHEM_PARTS = pd.DataFrame(
        [{"part": f"angle loader {s['slot']}", "file": s["dataset_path"], "sha256": s["file_sha256"][:16] + "…",
          "published by": s["author"], "version": s.get("notebook_version", ""),
          "Toffolis per shot": s.get("toffoli_per_shot_source"), "link": s["notebook"]} for s in ASSEMBLY["slots"]]
        + [{"part": "weights (submission.npz)", "file": ASSEMBLY["npz"]["dataset_path"],
            "sha256": ASSEMBLY["npz"]["sha256"][:16] + "…", "published by": ASSEMBLY["npz"]["author"],
            "version": ASSEMBLY["npz"].get("notebook_version", ""), "Toffolis per shot": None,
            "link": ASSEMBLY["npz"]["notebook"]},
           {"part": f"word table {ASSEMBLY['table_key']} (origin)", "file": "parts/word_tables.csv",
            "sha256": "", "published by": ASSEMBLY["word_table_origin"]["author"], "version": "",
            "Toffolis per shot": None, "link": ASSEMBLY["word_table_origin"]["notebook"]}])
    show(CHEM_PARTS)
# G3 compares this run's verifier with the source record, so it covers only the picks verified this run; a
# reused pick's cost IS its source record (named in the 'verification' column above).
ran = [t for t in REFERENCE if not results[t].get("reused")]
rel = {t: abs(results[t]["result"]["score"] / float(picks[t]["local_task_score"]) - 1) for t in ran}
G3_NAME = f"G3 every cost verified this run within 1e-9 of its source record ({len(ran)}/3 verified this run)"
GATES = {
    "G1 every file matches its listed SHA-256": True,  # asserted when the files were assembled
    "G2 all three passed (this run: kickmix-engine or the official verifier; or an earlier official-verifier report for the exact bytes)":
        all(r["result"]["status"] == "PASS" for r in results.values()),
    G3_NAME: max(rel.values(), default=0.0) <= 1e-9,
    "G4 notebook wall time under 2 hours": WALL <= 7200,
}
show(pd.DataFrame([{"gate": g, "result": "pass" if ok else "FAIL"} for g, ok in GATES.items()]))
# G3: each cost reproduced exactly at the default salt (the local verifier seeds every shot from the file's
# SHA-256). This run's default-salt GM is one DQI draw; the grader draws with its own salt, so the headline's
# error bounds (expected, lucky, unlucky) are the fair summary, not this single draw.

# Headroom: how close each pick is to the verifier's pass/fail limits (from this run's verifier output)
E1_MAX = E2_MAX = 0.020          # Ha, both chemistry accuracy gates
GAP_FLOOR = 3.914898             # strict E_gap floor in the dfthcsos verifier
head = []
inv = results["invsqrt"]["result"]
w = inv.get("worst_error_over_tolerance")
if w is not None:
    head.append({"task": TASK_NAMES["invsqrt"], "check": "worst error / tolerance (must stay below 1)",
                 "value": f"{w:.5f}", "margin": f"{(1 - w) * 100:.3f}% of tolerance",
                 "tight": (1 - w) < 0.005})
dq = results["dqi"]["result"]
if results["dqi"].get("reused"):  # a reused report carries the cost only; the 48-draw risk is in the headline
    head.append({"task": TASK_NAMES["dqi"], "check": "failed shots on this file's draw (value / phase)",
                 "value": "unavailable (reused report)", "margin": "see the headline's error bounds",
                 "tight": False})
else:
    head.append({"task": TASK_NAMES["dqi"], "check": "failed shots on this file's draw (value / phase)",
                 "value": f"{dq.get('value_errors')} / {dq.get('phase_errors')}",
                 "margin": f"repetition factor R = {dq.get('repetition_factor', float('nan')):.5f}; "
                           "each extra failure costs about 0.46% of DQI cost",
                 "tight": (dq.get("value_errors") or 0) + (dq.get("phase_errors") or 0) > 0})
ch = results["chemistry"]["result"].get("metrics", {})
for key, lim in (("error_1", E1_MAX), ("error_2", E2_MAX)):
    if key in ch:
        m = (lim - ch[key]) * 1000
        head.append({"task": TASK_NAMES["chemistry"], "check": f"{key} (must stay ≤ 20 mHa)",
                     "value": f"{ch[key] * 1000:.5f} mHa", "margin": f"{m:.5f} mHa", "tight": m < 0.01})
if "E_gap" in ch:
    m = ch["E_gap"] - GAP_FLOOR
    head.append({"task": TASK_NAMES["chemistry"], "check": "E_gap (must stay above 3.914898)",
                 "value": f"{ch['E_gap']:.7f}", "margin": f"{m:.2e}", "tight": m < 1e-4})
hd = pd.DataFrame(head)
hd["status"] = ["tight" if t else "comfortable" for t in hd.pop("tight")]
# Headroom: how close each pick is to failing the verifier; 'tight' means a tiny change in the file, the
# verifier or floating point could flip it.
show(hd)

# DQI final-salt risk: measured from the pick's 48-draw sweep (DQI_RISK, verify cell) and shown in the
# headline's error bounds; a DQI Salt Check output covering this file is kept in receipt.json when attached.
SALT = None
for f in sorted(glob.glob(f"{INPUT}/**/dqi_salt_check_summary_*.csv", recursive=True)):
    for r in csv.DictReader(open(f, newline="")):
        if r.get("circuit") == picks["dqi"]["sha12"]:
            SALT = dict(r, file=Path(f).name)
if not GATES[G3_NAME]:
    print("G3 failed: this run's verifier disagrees with a source record, so treat the local score with care.")
(WORK / "receipt.json").write_text(json.dumps({
    "gates": GATES, "verification": VERIFICATION, "geometric_mean_local": gm, "headroom": head, "dqi_salt_check": SALT, "dqi_risk": DQI_RISK, "wall_seconds": WALL, "relative_cost_difference": rel,
    "submission_sha256": sha256(WORK / "submission.zip"), "manifest_sha256": sha256(manifest),
    "rows": receipt.to_dict(orient="records"),
    "chemistry_parts": CHEM_PARTS.to_dict(orient="records") if CHEM_PARTS is not None else None},
    indent=1, default=str))